# recursive_opt — Use-Case Experiment Suite

This notebook runs **3 complementary experiments per use case** to maximize the chance
of a successful / informative result, measures them in a comparison table, and
displays the winning artifact/code so you can inspect and reuse it.

## Read this first: what "offline" really means here
A Trace **optimizer** (OptoPrimeV2) calls an LLM — so *genuine recursive optimization
requires an API key* (`LIVE = True` below). What runs **without** a key is:
- the **evaluator / plumbing** (scoring a candidate is deterministic for the code surface),
- an **offline plumbing pre-flight** that installs a *hand-written* improved candidate to
  prove the score is climbable and the evaluator works — it does **not** discover the
  improvement, it only validates the surface.

So: **offline = validate the surface & evaluator. live = actually optimize.**
Every experiment below declares which mode it needs. Set `LIVE=True` + a key for the
real thing; leave `LIVE=False` to dry-run the plumbing and inspect the specs.


In [1]:
# ============================ CONFIGURATION ===============================
# LIVE=True runs REAL recursive optimization. The model is configured here so
# every optimizer / Trace-Bench inference path uses the same backend.
import os, sys, json, time, statistics, textwrap, math
from pathlib import Path

# Make Run-All robust from repo root, examples/, or nbconvert kernels whose cwd
# is not automatically inserted on sys.path. This is notebook-only path setup; it
# does not change the installed package.
_REPO_ROOT = Path.cwd()
if not (_REPO_ROOT / "opto").exists() and (_REPO_ROOT.parent / "opto").exists():
    _REPO_ROOT = _REPO_ROOT.parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

LIVE = True
MODEL = os.environ.get("RECURSIVE_OPT_MODEL") or os.environ.get("TRACE_LITELLM_MODEL") or "gpt-5.4-nano"
os.environ["RECURSIVE_OPT_MODEL"] = MODEL
os.environ["TRACE_LITELLM_MODEL"] = MODEL

# Budget - these map 1:1 to spec["budget"] (RecursiveOptBudget). Tune per run.
WALL_TIME_S          = 1800  # hard wall-clock cap per run (~30 min); the simplest guard
RUN_ITERATIONS       = 2     # optimizer update steps per seed/run
NUM_CANDIDATES       = 2     # proposals per optimizer step
MAX_OPTIMIZER_CALLS  = 8     # caps LLM proposal cost per seed/run
MAX_EVAL_CALLS       = 48    # standard cap for prompt/config/code runs
CAPABILITY_EVAL_CALLS = 96  # UC3 does train + final eval over 8 examples; 48 exhausted before save_priors
MAX_CANDIDATES       = 8     # caps search breadth (iterations * num_candidates)
os.environ["RECURSIVE_OPT_ITERATIONS"] = str(RUN_ITERATIONS)
os.environ["RECURSIVE_OPT_NUM_CANDIDATES"] = str(NUM_CANDIDATES)

# Task-eval bounds dominate cost more than anything. 8 examples is the current
# default here because earlier 4-example runs saturated too easily and were noisy.
MAX_EXAMPLES = 8
HARD_MAX_EXAMPLES = int(os.environ.get("RECURSIVE_OPT_HARD_MAX_EXAMPLES", "4"))  # HF QA tasks are slower; 4 reduces single-example noise without making Run-All impractical
INNER_STEPS  = 0             # 0 = artifact-only (fast); >0 = real inner training (costly)
TIMEOUT_S    = 35            # per-eval timeout
os.environ["RECURSIVE_OPT_CAPABILITY_MAX_EXAMPLES"] = str(MAX_EXAMPLES)

# Two seeds are the minimum useful repeated live check; add seed 2 for publication runs.
SEEDS = [0, 1]

# Keep every generated memory/artifact folder under one run directory.
# If the notebook is launched from the repo root, this resolves to
# examples/notebook_outputs/...; if launched from examples/, it resolves to
# notebook_outputs/... under examples/. Override with RECURSIVE_OPT_OUTPUT_ROOT.
RUN_ID = os.environ.get("RECURSIVE_OPT_RUN_ID") or time.strftime("use_cases_%Y%m%d_%H%M%S")
_DEFAULT_OUTPUT_ROOT = (Path("examples/notebook_outputs/recursive_opt_use_cases")
                        if Path("examples").exists()
                        else Path("notebook_outputs/recursive_opt_use_cases"))
OUTPUT_ROOT = Path(os.environ.get("RECURSIVE_OPT_OUTPUT_ROOT", str(_DEFAULT_OUTPUT_ROOT))) / RUN_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# credit_horizon controls how much per-example guide feedback the optimizer sees.
# For UC6 we fix it at the previously best setting ("step") and compare trace designs.

# NOTE on the hf:gsm8k alias: recursive_opt currently redirects hf:gsm8k ->
# internal:multiobjective_gsm8k. We use internal:* families directly to avoid ambiguity.
if LIVE:
    from opto.features.recursive_opt.runmode import preflight_model
    from opto.features.recursive_opt.tracebench import ensure_default_task_adapter
    preflight_model(MODEL)
    ensure_default_task_adapter(require=True)
print("LIVE =", LIVE, "| model =", MODEL,
      "| iterations =", RUN_ITERATIONS, "| candidates =", NUM_CANDIDATES,
      "| examples =", MAX_EXAMPLES, "| hard_examples =", HARD_MAX_EXAMPLES, "| seeds =", SEEDS,
      "| eval_calls =", MAX_EVAL_CALLS, "| capability_eval_calls =", CAPABILITY_EVAL_CALLS,
      "| output_root =", OUTPUT_ROOT)


LIVE = True | model = gpt-5.4-nano | iterations = 2 | candidates = 2 | examples = 8 | hard_examples = 4 | seeds = [0, 1] | eval_calls = 48 | capability_eval_calls = 96 | output_root = notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614


In [2]:
# ============================ DRY HARNESS ================================
# One place that runs a spec, collects initial/final scores and artifact refs
# across seeds, and renders comparison tables. Every use case reuses this.
from opto.features.recursive_opt import run_spec, make_level_spec, MemoryLite
from opto.features.recursive_opt.budget import RecursiveOptBudget, reset_budget


def budget_block():
    """Standard budget dict from the config knobs above (spec['budget'] keys)."""
    return {"wall_time_s": WALL_TIME_S, "optimizer_llm_calls": MAX_OPTIMIZER_CALLS,
            "eval_llm_calls": MAX_EVAL_CALLS, "candidates": MAX_CANDIDATES,
            "on_exceed": "return_best"}


def make_budget():
    """Finite budget for direct optimize() calls outside run_spec."""
    return RecursiveOptBudget(
        max_wall_time_s=WALL_TIME_S,
        max_optimizer_llm_calls=MAX_OPTIMIZER_CALLS,
        max_eval_llm_calls=MAX_EVAL_CALLS,
        max_candidates=MAX_CANDIDATES,
        stop_policy="return_best",
    )


def reset_standard_budget():
    """Reset direct optimize() calls to the same envelope as spec runs."""
    reset_budget(make_budget())


def memory_path(name):
    """Return an experiment memory path under the common OUTPUT_ROOT."""
    safe = str(name).strip().strip("./") or "mem"
    return str(OUTPUT_ROOT / safe)


def tracebench_block(max_examples=None, inner_steps=None, timeout_seconds=None, eval_kwargs=None):
    """Standard real-adapter bounds (spec['tracebench'] keys).

    Keep this centralized so hard-task experiments can use fewer examples without
    changing the global notebook budget or relying on environment variables.
    """
    block = {"max_examples": int(max_examples or MAX_EXAMPLES),
             "inner_steps": INNER_STEPS if inner_steps is None else int(inner_steps),
             "timeout_seconds": TIMEOUT_S if timeout_seconds is None else int(timeout_seconds)}
    if eval_kwargs:
        block["eval_kwargs"] = dict(eval_kwargs)
    return block


def _one_line_error(exc):
    """Compact, table-safe error summary using the first non-empty message line."""
    lines = [line.strip() for line in str(exc).splitlines() if line.strip()]
    detail = lines[0][:160] if lines else repr(exc)[:160]
    return f"{type(exc).__name__}: {detail}"


def _finite(values):
    """Return finite float values only."""
    out = []
    for value in values:
        try:
            f = float(value)
        except (TypeError, ValueError):
            continue
        if math.isfinite(f):
            out.append(f)
    return out


def _fmt(value):
    """Format optional numeric values for markdown tables."""
    if value is None:
        return "-"
    try:
        f = float(value)
    except (TypeError, ValueError):
        return str(value)
    return f"{f:.3f}" if math.isfinite(f) else "-"


def _md_cell(value: object) -> str:
    """Escape text for a single markdown table cell."""
    text = "-" if value is None else str(value)
    return text.replace("\n", "<br>").replace("|", "\\|")


def _md_code(value: object) -> str:
    """Render a markdown table cell as inline code without breaking pipes."""
    text = _md_cell(value).replace("`", "\\`")
    return f"`{text}`"


def _compact_markdown_tables(markdown: str) -> str:
    """Remove blank lines that would terminate an active markdown table."""
    lines = markdown.splitlines()
    compacted = []
    for index, line in enumerate(lines):
        if line.strip() == "" and compacted and compacted[-1].lstrip().startswith("|"):
            next_index = index + 1
            while next_index < len(lines) and lines[next_index].strip() == "":
                next_index += 1
            if next_index < len(lines) and lines[next_index].lstrip().startswith("|"):
                continue
        compacted.append(line)
    return "\n".join(compacted)


def _display_markdown(markdown: str) -> None:
    """Display markdown after applying table-safety normalization."""
    display(Markdown(_compact_markdown_tables(markdown)))


def _artifact_file(root):
    """Return the JSONL file where reusable artifacts are persisted."""
    return str(Path(root) / "artifacts.jsonl")


def _artifact_ref(root, artifact_id=None):
    """Reference the persisted artifact by file plus optional artifact id."""
    path = _artifact_file(root)
    return f"{path}#{artifact_id}" if artifact_id else path


def _result_mean(result):
    """Mean final score for a result dict, or None when no run succeeded."""
    scores = _finite(result.get("scores", []))
    return statistics.mean(scores) if scores else None


def initial_score_for_spec(spec, level_id=None, run_name=None):
    """Evaluate the unoptimized seed artifact once, using the same real adapter bounds."""
    if not LIVE:
        return None, None
    import opto.features.recursive_opt.spec as spec_mod
    try:
        reset_standard_budget()
        lid = level_id or spec["levels"][-1]["id"]
        base_root = Path(run_name or spec.get("memory_root", "mem")).name
        probe_spec = {**spec, "memory_root": memory_path(f"_initial_probes/{base_root}")}
        spec_mod.validate_spec(probe_spec)
        if "tracebench" in probe_spec:
            from opto.features.recursive_opt import tracebench as TB
            TB.configure_tracebench_adapter(probe_spec.get("tracebench") or {}, require=True)
        families = probe_spec.get("families", {})
        memory = MemoryLite(root=probe_spec["memory_root"])
        for level_spec in probe_spec["levels"]:
            level = spec_mod.compile_level(level_spec, memory, families, probe_spec.get("scoring"))
            if level_spec["id"] == lid:
                score, _data = spec_mod._final_eval(level, level_spec, families)
                score = spec_mod._clamp(score, spec_mod._clip_bounds(probe_spec.get("scoring")))
                return float(score), None
        return None, f"level {lid!r} not found"
    except Exception as exc:
        return None, _one_line_error(exc)


def run_spec_seeds(spec, seeds=SEEDS, level_id=None, run_name=None):
    """Run a spec across seeds and keep failures visible without stopping the suite."""
    scores, walls, artifact, aid, errors = [], [], None, None, []
    artifact_ref, best_score = None, None
    lid = level_id or spec["levels"][-1]["id"]
    base_root = Path(run_name or spec.get("memory_root", "mem")).name
    if not LIVE:
        from opto.features.recursive_opt import validate_spec
        validate_spec(spec)
        return {"scores": [], "initial": None, "wall_s": None,
                "artifact": "(dry-run: set LIVE=True to optimize)",
                "artifact_id": None, "artifact_file": None, "dry": True}

    initial, initial_error = initial_score_for_spec(spec, level_id=lid, run_name=base_root)
    if initial_error:
        errors.append(f"initial: {initial_error}")
    for seed in seeds:
        try:
            reset_standard_budget()
            root = memory_path(f"{base_root}_{seed}")
            out = run_spec({**spec, "memory_root": root})
            r = out["results"][lid]
            score = float(r["score"])
            scores.append(score); walls.append(float(r["wall_s"]))
            ref = _artifact_ref(root, r.get("artifact_id"))
            if best_score is None or score > best_score:
                best_score = score
                artifact, aid, artifact_ref = r["artifact"], r.get("artifact_id"), ref
        except Exception as exc:
            errors.append(f"seed {seed}: {_one_line_error(exc)}")
    return {"scores": scores, "initial": initial,
            "wall_s": round(statistics.mean(walls), 1) if walls else None,
            "artifact": artifact or "(no successful seed)", "artifact_id": aid,
            "artifact_file": artifact_ref, "errors": errors, "dry": False}


def _notes_for_result(result: dict[str, object]) -> str:
    """Return compact interpretation and error notes for result tables."""
    notes = []
    if result.get("control_reason"):
        notes.append(str(result["control_reason"]))
    errors = list(result.get("errors", []) or [])
    notes.extend(str(error) for error in errors[:2])
    if len(errors) > 2:
        notes.append(f"+{len(errors)-2} more")
    return "; ".join(notes)


def summarize(rows):
    """rows: list of (label, result_dict). Returns a markdown comparison table."""
    head = ("| experiment | initial | mean score | delta | std | n | wall_s | best artifact file | notes |\n"
            "|---|---:|---:|---:|---:|---:|---:|---|---|")
    lines = [head]
    for label, r in rows:
        if r.get("dry"):
            lines.append(f"| {_md_cell(label)} | - | dry-run | - | - | 0 | - | - | set LIVE=True |")
            continue
        scores = _finite(r.get("scores", []))
        mean = statistics.mean(scores) if scores else None
        std = statistics.pstdev(scores) if len(scores) > 1 else None
        delta = (mean - r["initial"]) if mean is not None and r.get("initial") is not None else None
        notes = _notes_for_result(r)
        artifact_file = r.get("artifact_file") or "-"
        lines.append(f"| {_md_cell(label)} | {_fmt(r.get('initial'))} | {_fmt(mean)} | {_fmt(delta)} | "
                     f"{_fmt(std)} | {len(scores)} | {_fmt(r.get('wall_s'))} | {_md_code(artifact_file)} | {_md_cell(notes)} |")
    return "\n".join(lines)


def mark_control(result: dict[str, object], reason: str) -> dict[str, object]:
    """Mark a valid experiment as a diagnostic/control rather than a best-arm candidate."""
    out = dict(result)
    out["exclude_best"] = True
    out["control_reason"] = reason
    return out


def best_of(rows):
    """Return the most informative best row: mean score, then gain, then lower wall time."""
    scored = [(l, r) for l, r in rows if _result_mean(r) is not None]
    informative = [(l, r) for l, r in scored if not r.get("exclude_best")]
    if informative:
        scored = informative
    if not scored:
        return None
    def key(row):
        _label, result = row
        mean = _result_mean(result)
        initial = result.get("initial")
        delta = mean - initial if mean is not None and initial is not None else 0.0
        return (mean, delta, -(result.get("wall_s") or 1e9))
    return max(scored, key=key)


from IPython.display import Markdown, display

def show_table(title, rows):
    display(Markdown(f"### {title}\n" + summarize(rows)))
    b = best_of(rows)
    if b:
        _display_markdown(f"**Best: `{_md_cell(b[0])}`** — artifact file: {_md_code(b[1].get('artifact_file') or '-')}")
        print(textwrap.shorten(str(b[1]["artifact"]), 1200, placeholder=" ...[truncated]"))


def _read_jsonl(path):
    """Read a JSONL file defensively for cross-run summaries."""
    p = Path(path)
    if not p.exists():
        return []
    rows = []
    for line in p.read_text().splitlines():
        if line.strip():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return rows


def _best_artifact_from_dir(mem_dir):
    """Best finite artifact record in a MemoryLite directory."""
    records = _read_jsonl(Path(mem_dir) / "artifacts.jsonl")
    valid = [r for r in records if _finite([r.get("score")])]
    return max(valid, key=lambda r: float(r["score"])) if valid else None


def _initial_from_dir(mem_dir):
    """Best-effort initial score from persisted artifact/episode records."""
    for filename in ("artifacts.jsonl", "episodes.jsonl"):
        records = _read_jsonl(Path(mem_dir) / filename)
        for record in records:
            metrics = record.get("metrics") if isinstance(record, dict) else None
            if isinstance(metrics, dict):
                history = metrics.get("score_history")
                if isinstance(history, list):
                    scores = _finite(history[:1])
                    if scores:
                        return scores[0]
                scores = _finite([metrics.get("initial")])
                if scores:
                    return scores[0]
            scores = _finite([record.get("score")])
            if scores:
                return scores[0]
    return None


UC_DIR_PREFIXES = {
    "UC1 component code": "mem_uc1",
    "UC2 setup/config": "mem_uc2",
    "UC3 capability": "mem_uc3",
    "UC4 family/transfer": "mem_uc4",
    "UC5 optimizer/tool": "mem_uc5",
    "UC6 trace feedback": "mem_uc6",
    "UC7 graph/suboptimizer": "mem_suboptimizer_graph",
}


def summarize_past_runs(base_dir=None):
    """Scan previous notebook output folders and summarize persisted artifact scores."""
    base = Path(base_dir or OUTPUT_ROOT.parent)
    rows = []
    for run_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        for uc_name, prefix in UC_DIR_PREFIXES.items():
            mem_dirs = sorted(p for p in run_dir.glob(f"{prefix}*") if p.is_dir())
            if not mem_dirs:
                continue
            best, best_dir = None, None
            finals, initials = [], []
            for mem_dir in mem_dirs:
                init = _initial_from_dir(mem_dir)
                if init is not None:
                    initials.append(init)
                art = _best_artifact_from_dir(mem_dir)
                if art is None:
                    continue
                score = float(art["score"])
                finals.append(score)
                if best is None or score > float(best["score"]):
                    best, best_dir = art, mem_dir
            if best is None:
                continue
            rows.append({
                "run": run_dir.name,
                "use_case": uc_name,
                "initial_mean": statistics.mean(initials) if initials else None,
                "best_score": float(best["score"]),
                "final_mean": statistics.mean(finals) if finals else None,
                "n_dirs": len(mem_dirs),
                "artifact_file": _artifact_ref(best_dir, best.get("artifact_id")),
            })
    return rows




def _experiment_from_mem_dir(mem_dir):
    """Compact experiment label inferred from a persisted MemoryLite directory."""
    name = Path(mem_dir).name
    parts = name.split("_")
    if parts and parts[-1].isdigit():
        name = "_".join(parts[:-1])
    return name


def summarize_past_experiments(base_dir=None):
    """Scan every past memory folder, not only the best use-case aggregate."""
    base = Path(base_dir or OUTPUT_ROOT.parent)
    rows = []
    for run_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        for uc_name, prefix in UC_DIR_PREFIXES.items():
            for mem_dir in sorted(p for p in run_dir.glob(f"{prefix}*") if p.is_dir()):
                art = _best_artifact_from_dir(mem_dir)
                if art is None:
                    continue
                rows.append({
                    "run": run_dir.name,
                    "use_case": uc_name,
                    "experiment": _experiment_from_mem_dir(mem_dir),
                    "initial": _initial_from_dir(mem_dir),
                    "best_score": float(art["score"]),
                    "artifact_file": _artifact_ref(mem_dir, art.get("artifact_id")),
                })
    return rows


def past_experiments_table(rows, limit=None):
    """Render every persisted experiment across all past notebook runs."""
    head = "| run | use case | experiment | initial | best score | best artifact file |\n|---|---|---|---:|---:|---|"
    lines = [head]
    ordered = sorted(rows, key=lambda r: (r["run"], r["use_case"], r["experiment"]))
    selected = ordered if limit is None else ordered[-limit:]
    for row in selected:
        lines.append(f"| {_md_cell(row['run'])} | {_md_cell(row['use_case'])} | {_md_cell(row['experiment'])} | "
                     f"{_fmt(row['initial'])} | {_fmt(row['best_score'])} | {_md_code(row['artifact_file'])} |")
    if limit is not None and len(ordered) > limit:
        lines.append(f"| ... | ... | {len(ordered)-limit} older rows omitted | - | - | - |")
    return "\n".join(lines)

def past_runs_table(rows):
    """Render the cross-run artifact summary."""
    head = "| run | use case | initial mean | final mean | best score | n dirs | best artifact file |\n|---|---|---:|---:|---:|---:|---|"
    lines = [head]
    for row in rows:
        lines.append(f"| {_md_cell(row['run'])} | {_md_cell(row['use_case'])} | {_fmt(row['initial_mean'])} | "
                     f"{_fmt(row['final_mean'])} | {_fmt(row['best_score'])} | {row['n_dirs']} | "
                     f"{_md_code(row['artifact_file'])} |")
    return "\n".join(lines)

print("harness ready")


harness ready


---
### Root-cause diagnostics
These quick checks separate optimizer behavior from benchmark shape. They are intentionally small: probe score spread tells us whether a surface has enough room to learn, while code-baseline probes show when UC1/UC5 are saturated by a narrow deterministic validator rather than by broad benchmark performance. BBEH is probed here for task-shape awareness, but it is not used in UC3 because its Trace-Bench bundle is raw/code-artifact style rather than a prompt-capability surface. The current O1 config surface binds one concrete task id; mixed easy+hard family learning is therefore represented by O2/O3, not by pretending one O1 row averages multiple unrelated tasks.

In [3]:
from opto.features.recursive_opt.spec import score_spread
from opto.features.recursive_opt.tracebench import make_code_evaluator, configure_tracebench_adapter

if LIVE:
    configure_tracebench_adapter(tracebench_block(), require=True)
    diagnostic_rows = []
    probe_prompts = [
        {},
        {"starting_artifact": "Answer directly."},
        {"starting_artifact": "Plan step by step, then verify the answer before replying."},
    ]
    for task in ["internal:multiobjective_gsm8k", "internal:multiobjective_bbeh", "hf:drop", "hf:qasper"]:
        try:
            spread = score_spread(task, probes=probe_prompts)
            scores = [r.get("score") for r in spread["rows"]]
            diagnostic_rows.append((task, spread["valid_spread"], spread["invalid_probes"], scores))
        except Exception as exc:
            diagnostic_rows.append((task, None, None, _one_line_error(exc)))

    code_probe = make_code_evaluator("internal:batch_design", "batch_design")
    code_rows = []
    for label, fn in [
        ("take_first", lambda n, k: list(range(k))),
        ("take_last", lambda n, k: list(range(n-k, n))),
        ("stride", lambda n, k: list(range(0, n, max(1, n//k)))[:k]),
        ("hard_mod3", lambda n, k: [i for i in range(n) if i % 3 == 0][:k]),
    ]:
        score, feedback = code_probe(lambda **kw: fn(**kw), "internal:batch_design")
        code_rows.append((label, score, feedback))

    lines = ["| probe | spread/score | details |", "|---|---:|---|"]
    for task, spread, invalid, scores in diagnostic_rows:
        lines.append(f"| {task} score spread | {_fmt(spread)} | invalid={invalid}; scores={scores} |")
    for label, score, feedback in code_rows:
        lines.append(f"| batch_design baseline `{label}` | {_fmt(score)} | {feedback[:180]} |")
    display(Markdown("\n".join(lines)))
else:
    display(Markdown("Diagnostics skipped: set `LIVE=True`."))


/home/xav/miniconda3/envs/humanllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


| probe | spread/score | details |
|---|---:|---|
| internal:multiobjective_gsm8k score spread | 0.041 | invalid=0; scores=[-0.15875, -0.11812500000000001, -0.1455] |
| internal:multiobjective_bbeh score spread | 1.000 | invalid=0; scores=[0.999989553625, -2.929574999988027e-06, -5.114624999924544e-06] |
| hf:drop score spread | 0.250 | invalid=0; scores=[1.0, 0.75, 1.0] |
| hf:qasper score spread | 0.073 | invalid=0; scores=[0.24460237829803047, 0.21943656833362715, 0.2919906477106093] |
| batch_design baseline `take_first` | 0.800 | [batch_design@internal:batch_design] validation_pool n=12, k=4; hard/failing indices are [0, 3, 6, 9] (defined by idx % 3 == 0); picked [0, 1, 2, 3]; hard_items=2/4; diversity=1.00 |
| batch_design baseline `take_last` | 0.700 | [batch_design@internal:batch_design] validation_pool n=12, k=4; hard/failing indices are [0, 3, 6, 9] (defined by idx % 3 == 0); picked [8, 9, 10, 11]; hard_items=1/4; diversity=1. |
| batch_design baseline `stride` | 1.000 | [batch_design@internal:batch_design] validation_pool n=12, k=4; hard/failing indices are [0, 3, 6, 9] (defined by idx % 3 == 0); picked [0, 3, 6, 9]; hard_items=4/4; diversity=1.00 |
| batch_design baseline `hard_mod3` | 1.000 | [batch_design@internal:batch_design] validation_pool n=12, k=4; hard/failing indices are [0, 3, 6, 9] (defined by idx % 3 == 0); picked [0, 3, 6, 9]; hard_items=4/4; diversity=1.00 |


---
## Use Case 1 — Optimize & validate a NEW Trace component (code surface) ⭐ strongest today

**Why:** the code surface has a deterministic evaluator, so the signal is clean and the
before/after is a real diff. Best for: a new trainer hot-path, batch sampler, trace
summarizer, validator, or compact task-solving component.

**Experiments:**
1. **batch_design** on `internal:batch_design` — known-climbable failure-balanced selector.
2. **trace_summarizer** on `internal:code_param` — multi-criteria evaluator (keep error evidence + be concise), with default and stricter prompt variants.
3. **BBEH direct code solver** on real Trace-Bench examples — harder than the toy selectors and saved as reusable Python code.

**Mode:** offline pre-flight proves the surface; **set LIVE=True for the real rewrite.**


In [4]:
# Use Case 1 — code surface. Uses ComponentSpec + CodeArtifactLevel via optimize().
# Interpretation: this proves optimizer-to-code rewriting, validation, rollback, and
# artifact persistence. It is intentionally narrow; saturation means the validator is
# easy, not that the learned component generalizes to all training loops.
from opto.features.recursive_opt import CodeArtifactLevel, ComponentSpec, optimize, RecursiveGuide
from opto.features.recursive_opt.tracebench import make_code_evaluator, make_dataset, make_tracebench_direct_answer_evaluator
import opto.trace as trace


def run_code_experiment(name, task_id, objective, seeds=SEEDS, memory_name=None, baseline=None, evaluate=None):
    """One code-surface experiment across isolated memory roots per seed."""
    if not LIVE:
        return {"scores": [], "initial": None, "wall_s": None,
                "artifact": f"(dry-run) component='{name}' task='{task_id}'; set LIVE=True to optimize",
                "artifact_id": None, "artifact_file": None, "dry": True}
    scores, initial_scores, walls, final_code, errors = [], [], [], None, []
    best_ref, best_score, best_code = None, None, None
    for seed in seeds:
        try:
            root_name = memory_name or f"mem_uc1_{name}"
            root = memory_path(f"{root_name}_{seed}")
            mem = MemoryLite(root=root)
            spec = ComponentSpec(name=name, baseline=baseline or _BASELINES[name],
                                 evaluate=evaluate or make_code_evaluator(task_id, name), objective=objective)
            level = CodeArtifactLevel(spec, memory=mem)
            guide = RecursiveGuide()
            initial_scores.append(float(guide(task_id, level.forward(task_id), None)[0]))
            reset_standard_budget()
            t0 = time.time()
            optimize(level, make_dataset([task_id], repeats=MAX_EXAMPLES), guide=guide,
                     iterations=RUN_ITERATIONS, num_candidates=NUM_CANDIDATES)
            # Code surfaces persist every validated implementation. Report and
            # re-score the best saved artifact so the table points at the reusable
            # solution, even if the Trainer's active slot moved on.
            best = mem.best_artifact(str(task_id), "code")
            if best is not None and level.parameters():
                level.parameters()[0]._data = best.content
            walls.append(round(time.time() - t0, 1))
            score = float(guide(task_id, level.forward(task_id), None)[0])
            if best is not None and float(best.score) >= score:
                score, final_code = float(best.score), best.content
                ref = _artifact_ref(root, best.artifact_id)
            else:
                final_code = level.current_code()
                ref = _artifact_file(root)
            scores.append(score)
            if best_score is None or score > best_score:
                best_score, best_ref, best_code = score, ref, final_code
        except Exception as exc:
            errors.append(f"seed {seed}: {_one_line_error(exc)}")
    return {"scores": scores, "initial": statistics.mean(initial_scores) if initial_scores else None,
            "wall_s": round(statistics.mean(walls), 1) if walls else None,
            "artifact": best_code or final_code or "(no successful seed)", "artifact_id": None,
            "artifact_file": best_ref, "errors": errors, "dry": False}


# Baselines kept deliberately weak so there is headroom to climb.
def _weak_batch(self, n, k): return list(range(k))
def _trunc_summary(self, trace_text): return str(trace_text)[-500:]
def _bbeh_direct_solver(self, question):
    """Return True/False for a BBEH boolean expression ending with ' is'."""
    return "True"

def _norm_bool_answer(value):
    """Normalize boolean answers for BBEH direct-solver validation."""
    return str(value).strip().lower().replace(".", "").replace(" ", "")

_BASELINES = {"batch_design": _weak_batch, "trace_summarizer": _trunc_summary,
              "bbeh_direct_solver": _bbeh_direct_solver}

uc1 = [
  ("batch_design (failure-balanced)",
   run_code_experiment("batch_design", "internal:batch_design",
                       "Select the hard/failing items before easy ones; maximize validator score.")),
  ("trace_summarizer (default)",
   run_code_experiment("trace_summarizer", "internal:code_param",
                       "Preserve failing-assertion evidence while removing noise; be concise.",
                       memory_name="mem_uc1_trace_summarizer_default")),
  ("trace_summarizer (strict)",
   run_code_experiment("trace_summarizer", "internal:code_param",
                       "Keep ALL error evidence, drop everything else, target <60 chars.",
                       memory_name="mem_uc1_trace_summarizer_strict")),
  ("BBEH direct code solver (real hard examples)",
   run_code_experiment("bbeh_direct_solver", "internal:multiobjective_bbeh",
                       "Rewrite the Python function to parse BBEH boolean expressions. Input `question` ends with ' is'. Return exactly 'True' or 'False'.",
                       memory_name="mem_uc1_bbeh_direct_solver",
                       evaluate=make_tracebench_direct_answer_evaluator(
                           "internal:multiobjective_bbeh", max_examples=MAX_EXAMPLES,
                           normalizer=_norm_bool_answer))),
]
show_table("Use Case 1 — component code optimization", uc1)


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:00<00:00, 2284.48it/s]

Evaluating agent:   0%|          | 0/8 [00:00<?, ?it/s]

Evaluating agent: 100%|██████████| 8/8 [00:00<00:00, 35320.45it/s]

...

Evaluating agent: 100%|██████████| 8/8 [00:00<00:00, 28581.29it/s]

[Step 1] Test/test_score: 1.0
[Step 1] Algo/Average train score: 0.71875
[Step 1] Update/n_iters: 1
[Step 1] Update/short_term_memory_size: 0
[Step 1] Update/long_term_memory_size: 3
[Step 1] Update/using_short_term_memory: False
[Step 1] Update/using_long_term_memory: True
[Step 1] Update/total_samples: 4
[Step 1] Update/best_candidate_priority: 1.0
[Step 1] Update/best_candidate_mean_score: 1.0
[Step 1] Update/best_candidate_num_rollouts: 1
[Step 1] Update/num_exploration_candidates: 2
[Step 1] Update/exploration_candidates_mean_priority: 0.8125
[Step 1] Update/exploration_candidates_mean_score: 0.8125
[Step 1] Update/exploration_candidates_average_num_rollouts: 1.5
[Step 1] Sample/mean_score: 0.8125
[Step 1] Sample/num_samples: 2
[Step 1] Sample/self.n_epochs: 0
[Step 1] Algo/Number of training samples: 4
[Step 1] Parameter/__code:8: def _bbeh_direct_solver(self, question):
    """Return True/False for a BBEH boolean expression ending with ' is'."""
    expr = question.strip()
    i

### Use Case 1 — component code optimization
| experiment | initial | mean score | delta | std | n | wall_s | best artifact file | notes |
|---|---:|---:|---:|---:|---:|---:|---|---|
| batch_design (failure-balanced) | 0.800 | 1.000 | 0.200 | 0.000 | 2 | 3.500 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:30451` |  |
| trace_summarizer (default) | 0.750 | 0.897 | 0.147 | 0.021 | 2 | 3.200 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc1_trace_summarizer_default_1/artifacts.jsonl#internal:code_param:code:1:40708` |  |
| trace_summarizer (strict) | 0.750 | 0.897 | 0.147 | 0.021 | 2 | 4.100 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc1_trace_summarizer_strict_0/artifacts.jsonl#internal:code_param:code:1:44914` |  |
| BBEH direct code solver (real hard examples) | 0.625 | 1.000 | 0.375 | 0.000 | 2 | 4.900 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc1_bbeh_direct_solver_0/artifacts.jsonl#internal:multiobjective_bbeh:code:1:54006` |  |


**Best: `BBEH direct code solver (real hard examples)`** — artifact file: `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc1_bbeh_direct_solver_0/artifacts.jsonl#internal:multiobjective_bbeh:code:1:54006`


def _bbeh_direct_solver(self, question): """Return True/False for a BBEH boolean expression ending with ' is'.""" q = question.strip() if not q.endswith(" is"): return "False" expr = q[:-3].strip() # remove trailing " is" # Safe-ish evaluation: only allow True/False, not/and/or, parentheses, whitespace. allowed_chars = set("TrueFalsenotandor() \t\n") if any(ch not in allowed_chars for ch in expr): return "False" try: # Evaluate using Python boolean semantics. val = eval(expr, {"__builtins__": None}, {}) return "True" if bool(val) else "False" except Exception: return "False"


---
## Use Case 2 — Learn the best SETUP / default prompt for a family (config surface)

**Why:** optimize *which existing components & artifact* to use. Under `INNER_STEPS=0`
only **causally-active** fields move score: `starting_artifact`, `initial_knowledge`,
`trace_type`. (Trainer/batch only activate at `INNER_STEPS>0` — the contract enforces this.)

**Experiments:**
1. **GSM8K artifact menu** — search a small set of prompt strategies (incl. empty control arm).
2. **+ initial_knowledge / warm prior** — test whether extra setup context or saved priors improve the same prompt surface.
3. **harder QA controls** — compare DROP (often saturated) with QASPER (less saturated but noisier/slower).

**Mode:** needs LIVE + Trace-Bench (real task scores).


In [5]:
# Use Case 2 — config surface via run_spec. Active fields only (INNER_STEPS=0).
# Use a prompt-compatible Trace-Bench task: starting_artifact is a system prompt
# here, not a numeric parameter. Scores are real adapter scores over MAX_EXAMPLES.
FAMILY_TASK = "internal:multiobjective_gsm8k"
HARD_PROMPT_TASKS = {"drop": "hf:drop", "qasper": "hf:qasper"}
ART_MENU = ["", "Answer directly.", "Plan step by step, then answer.",
            "Plan step by step, then verify the answer before replying.",
            "Use the provided context as evidence, reason briefly, then answer exactly."]


def config_spec(targets, reuse=False, extra_constraints=None, memory_root="./mem_uc2",
                task=FAMILY_TASK, family_name="reasoning", max_examples=None):
    cons = {"starting_artifact": ART_MENU}
    if extra_constraints:
        cons.update(extra_constraints)
    return {"families": {family_name: [task]},
            "memory_root": memory_root, "reuse_priors": reuse,
            "budget": budget_block(), "tracebench": tracebench_block(max_examples=max_examples),
            "levels": [ make_level_spec(
                id="o1_setup", surface="config", family=family_name, task=task,
                targets=targets, constraints=cons,
                fixed={"optimizer": "OptoPrimeV2", "trace_type": "internal",
                       "credit_horizon": "step", "trainer": "PrioritySearch"},
                iterations=RUN_ITERATIONS)]}

uc2 = [
  ("artifact only",       run_spec_seeds(config_spec(["starting_artifact"], memory_root="./mem_uc2_artifact_only"),
                                         run_name="mem_uc2_artifact_only")),
  ("artifact+knowledge",  run_spec_seeds(config_spec(["starting_artifact", "initial_knowledge"],
                                                      memory_root="./mem_uc2_artifact_knowledge"),
                                         run_name="mem_uc2_artifact_knowledge")),
  ("artifact (warm prior)", run_spec_seeds(config_spec(["starting_artifact"], reuse=True,
                                                        memory_root="./mem_uc2_warm_prior"),
                                           run_name="mem_uc2_warm_prior")),
  ("artifact on DROP (QA control; often saturated)", mark_control(
      run_spec_seeds(config_spec(["starting_artifact"],
                                  memory_root="./mem_uc2_drop",
                                  task=HARD_PROMPT_TASKS["drop"],
                                  family_name="drop", max_examples=HARD_MAX_EXAMPLES),
                     run_name="mem_uc2_drop"),
      "saturated control: useful for comparison, not selected as best")),
  ("artifact on QASPER (harder QA)", run_spec_seeds(config_spec(["starting_artifact"],
                                                        memory_root="./mem_uc2_qasper",
                                                        task=HARD_PROMPT_TASKS["qasper"],
                                                        family_name="qasper", max_examples=HARD_MAX_EXAMPLES),
                                           run_name="mem_uc2_qasper")),
]
show_table("Use Case 2 — setup/config optimization", uc2)


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 2 agents on 1 inputs:  50%|█████     | 1/2 [00:11<00:11, 11.57s/it]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:12<00:00,  5.18s/it]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:12<00:00,  6.14s/it]

...

Evaluating agent: 100%|██████████| 2/2 [00:07<00:00,  3.68s/it]

[Step 1] Test/test_score: 0.15802294902354672
[Step 1] Algo/Average train score: -0.1228694266167535
[Step 1] Update/n_iters: 1
[Step 1] Update/short_term_memory_size: 0
[Step 1] Update/long_term_memory_size: 3
[Step 1] Update/using_short_term_memory: False
[Step 1] Update/using_long_term_memory: True
[Step 1] Update/total_samples: 4
[Step 1] Update/best_candidate_priority: 0.18203469481620385
[Step 1] Update/best_candidate_mean_score: 0.18203469481620385
[Step 1] Update/best_candidate_num_rollouts: 2
[Step 1] Update/num_exploration_candidates: 2
[Step 1] Update/exploration_candidates_mean_priority: -0.4089826525918981
[Step 1] Update/exploration_candidates_mean_score: -0.4089826525918981
[Step 1] Update/exploration_candidates_average_num_rollouts: 1.5
[Step 1] Sample/mean_score: -0.42777354804971085
[Step 1] Sample/num_samples: 2
[Step 1] Sample/self.n_epochs: 0
[Step 1] Algo/Number of training samples: 4
[Step 1] Parameter/level_config:14: starting_artifact: 


### Use Case 2 — setup/config optimization
| experiment | initial | mean score | delta | std | n | wall_s | best artifact file | notes |
|---|---:|---:|---:|---:|---:|---:|---|---|
| artifact only | -0.163 | -0.136 | 0.026 | 0.011 | 2 | 66.500 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_artifact_only_1/artifacts.jsonl#reasoning:config:0:30223` |  |
| artifact+knowledge | -0.156 | -0.161 | -0.005 | 0.004 | 2 | 65.600 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_artifact_knowledge_1/artifacts.jsonl#reasoning:config:0:719` |  |
| artifact (warm prior) | -0.162 | -0.134 | 0.028 | 0.018 | 2 | 65.100 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_warm_prior_0/artifacts.jsonl#reasoning:config:0:87661` |  |
| artifact on DROP (QA control; often saturated) | 0.750 | 0.875 | 0.125 | 0.125 | 2 | 36.300 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_drop_1/artifacts.jsonl#drop:config:0:60919` | saturated control: useful for comparison, not selected as best |
| artifact on QASPER (harder QA) | 0.113 | 0.163 | 0.049 | 0.020 | 2 | 39.900 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_qasper_1/artifacts.jsonl#qasper:config:0:62209` |  |


**Best: `artifact on QASPER (harder QA)`** — artifact file: `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_qasper_1/artifacts.jsonl#qasper:config:0:62209`


starting_artifact:


---
## Use Case 3 — Discover a new CAPABILITY from a spec + objectives (capability surface)

**Why:** synthesize a capability artifact (a skill-like text) that satisfies a
natural-language spec while trading off objectives (accuracy↑, cost↓).

**3 experiments:** three different **seed specifications** for the same objective set, to
see which framing the optimizer can push furthest (a complementary-results search).

**Mode:** needs LIVE + Trace-Bench. Uses the `capability` surface in a spec.

In [6]:
# Use Case 3 — capability surface. Three seed framings; pareto over (accuracy, cost).
# Keep this on prompt-compatible GSM8K. BBEH is intentionally excluded here: the
# diagnostic probe showed it is a raw/code-artifact task for this adapter, so a
# natural-language capability prompt is evaluated as invalid code. That belongs to
# code-artifact experiments, not this prompt-capability surface.
from opto.features.recursive_opt.tracebench import make_multiobjective_evaluator

CAP_TASKS = ["internal:multiobjective_gsm8k"]
CAP_OBJECTIVES = {"accuracy": "max", "cost": "min"}
_cap_evaluator = make_multiobjective_evaluator(
    CAP_TASKS,
    CAP_OBJECTIVES,
    required_terms=("plan", "verify"),
)


def capability_spec(seed_text, memory_root="./mem_uc3"):
    cap_budget = {**budget_block(), "eval_llm_calls": CAPABILITY_EVAL_CALLS}
    return {"families": {"reasoning": CAP_TASKS}, "memory_root": memory_root,
            "budget": cap_budget, "tracebench": tracebench_block(),
            "levels": [ make_level_spec(
                id="cap", surface="capability", family="reasoning", task=CAP_TASKS[0],
                seed=seed_text, evaluator=_cap_evaluator,
                objective_config={"mode": "pareto", "minimize": ["cost"]},
                iterations=RUN_ITERATIONS)]}

uc3 = [
  ("seed: terse",   run_spec_seeds(capability_spec("Solve correctly using the fewest words.", "./mem_uc3_terse"),
                                   run_name="mem_uc3_terse")),
  ("seed: verify",  run_spec_seeds(capability_spec("Make a short plan; solve; then verify/check the answer before replying.", "./mem_uc3_verify"),
                                   run_name="mem_uc3_verify")),
  ("seed: decompose", run_spec_seeds(capability_spec("Plan, decompose into sub-steps, solve each, then verify before answering.", "./mem_uc3_decompose"),
                                     run_name="mem_uc3_decompose")),
]
show_table("Use Case 3 — capability discovery", uc3)


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 2 agents on 1 inputs:  50%|█████     | 1/2 [00:11<00:11, 11.37s/it]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:11<00:00,  5.69s/it]

Evaluating agent:   0%|          | 0/2 [00:00<?, ?it/s]

...

Evaluating agent: 100%|██████████| 2/2 [00:11<00:00,  5.98s/it]

[Step 1] Test/test_score: 1.4391666666666667
[Step 1] Algo/Average train score: 1.3743750000000001
[Step 1] Update/n_iters: 1
[Step 1] Update/short_term_memory_size: 0
[Step 1] Update/long_term_memory_size: 3
[Step 1] Update/using_short_term_memory: False
[Step 1] Update/using_long_term_memory: True
[Step 1] Update/total_samples: 4
[Step 1] Update/best_candidate_priority: 1.4391666666666667
[Step 1] Update/best_candidate_mean_score: 1.4391666666666667
[Step 1] Update/best_candidate_num_rollouts: 2
[Step 1] Update/num_exploration_candidates: 2
[Step 1] Update/exploration_candidates_mean_priority: 0.94625
[Step 1] Update/exploration_candidates_mean_score: 1.3095833333333333
[Step 1] Update/exploration_candidates_average_num_rollouts: 1.5
[Step 1] Sample/mean_score: 1.3095833333333333
[Step 1] Sample/num_samples: 2
[Step 1] Sample/self.n_epochs: 0
[Step 1] Algo/Number of training samples: 4
[Step 1] Parameter/capability:8: Plan, decompose into sub-steps, solve each, then verify before ans

### Use Case 3 — capability discovery
| experiment | initial | mean score | delta | std | n | wall_s | best artifact file | notes |
|---|---:|---:|---:|---:|---:|---:|---|---|
| seed: terse | 0.968 | 0.878 | -0.090 | 0.090 | 2 | 53.200 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc3_terse_0/artifacts.jsonl#reasoning:capability:0:38889` |  |
| seed: verify | 1.441 | 1.378 | -0.062 | 0.062 | 2 | 63.600 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc3_verify_0/artifacts.jsonl#reasoning:capability:0:96998` |  |
| seed: decompose | 1.439 | 1.445 | 0.006 | 0.006 | 2 | 63.700 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc3_decompose_0/artifacts.jsonl#reasoning:capability:0:59196` |  |


**Best: `seed: decompose`** — artifact file: `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc3_decompose_0/artifacts.jsonl#reasoning:capability:0:59196`


Plan briefly, solve stepwise, then verify the final answer.


---
## Use Case 4 — Family policy (O2) & transferable prior (O3) — EXPERIMENTAL

**Why:** learn config per family (O2) and induce a prior validated on held-out families (O3).
This is mechanically real but still noisy: treat results as exploratory and require warm>cold
by more than run noise before believing transfer.

**Experiments:**
1. **O2 only** — one family-policy level over a small mixed task set.
2. **O2→O3 cold** — add prior induction with no prior reuse.
3. **O2→O3 warm** — re-run with prior reuse to measure transfer.

**Mode:** needs LIVE. The mixed task set intentionally includes non-saturated QASPER so transfer is not judged only on saturated controls.


In [7]:
# Use Case 4 — O2/O3 via multi-level spec. Prompt-compatible internal
# families keep starting_artifact transfer meaningful and bounded. If both cold
# and warm O3 saturate, the conclusion is that this task mix is too easy for
# transfer evidence, not that transfer is universally solved.
FAMS = {"gsm8k": ["internal:multiobjective_gsm8k"], "drop": ["hf:drop"], "qasper": ["hf:qasper"]}


def o2_spec(memory_root="./mem_uc4_o2"):
    return {"families": FAMS, "memory_root": memory_root,
            "budget": budget_block(), "tracebench": tracebench_block(max_examples=HARD_MAX_EXAMPLES),
            "scoring": {"mode": "relative_delta", "clip": [-1, 1]},
            "levels": [ make_level_spec(id="o2", surface="family_policy",
                families=list(FAMS), targets=["starting_artifact"],
                fixed={"trace_type": "internal", "credit_horizon": "step"},
                iterations=RUN_ITERATIONS)]}


def o2o3_spec(reuse=False, memory_root="./mem_uc4_o3"):
    return {"families": FAMS, "memory_root": memory_root, "reuse_priors": reuse,
            "budget": budget_block(), "tracebench": tracebench_block(max_examples=HARD_MAX_EXAMPLES),
            "scoring": {"mode": "relative_delta", "clip": [-1, 1]},
            "levels": [
              make_level_spec(id="o2", surface="family_policy", families=list(FAMS),
                  targets=["starting_artifact"], fixed={"trace_type":"internal", "credit_horizon":"step"},
                  iterations=RUN_ITERATIONS),
              make_level_spec(id="o3", surface="prior", families=list(FAMS),
                  targets=["starting_artifact"], fixed={"trace_type":"internal", "credit_horizon":"step"},
                  depends_on=["o2"], iterations=RUN_ITERATIONS)]}

uc4 = [
  ("O2 family policy",      run_spec_seeds(o2_spec("./mem_uc4_o2_policy"), level_id="o2", run_name="mem_uc4_o2_policy")),
  ("O2->O3 (cold)",         run_spec_seeds(o2o3_spec(reuse=False, memory_root="./mem_uc4_o3_cold"),
                                           level_id="o3", run_name="mem_uc4_o3_cold")),
  ("O2->O3 (warm prior)",   run_spec_seeds(o2o3_spec(reuse=True, memory_root="./mem_uc4_o3_warm"),
                                           level_id="o3", run_name="mem_uc4_o3_warm")),
]
show_table("Use Case 4 — family policy & transfer (experimental)", uc4)


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 2 agents on 1 inputs:  50%|█████     | 1/2 [00:41<00:41, 41.21s/it]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:44<00:00, 18.90s/it]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:44<00:00, 22.25s/it]

...

Evaluating agent: 100%|██████████| 2/2 [00:14<00:00,  7.04s/it]

[Step 1] Test/test_score: -0.0018419068525826338
[Step 1] Algo/Average train score: -0.013749622103307602
[Step 1] Update/n_iters: 1
[Step 1] Update/short_term_memory_size: 0
[Step 1] Update/long_term_memory_size: 3
[Step 1] Update/using_short_term_memory: False
[Step 1] Update/using_long_term_memory: True
[Step 1] Update/total_samples: 4
[Step 1] Update/best_candidate_priority: -0.025201685222691977
[Step 1] Update/best_candidate_mean_score: -0.025201685222691977
[Step 1] Update/best_candidate_num_rollouts: 1
[Step 1] Update/num_exploration_candidates: 2
[Step 1] Update/exploration_candidates_mean_priority: -0.02562521112935032
[Step 1] Update/exploration_candidates_mean_score: -0.02562521112935032
[Step 1] Update/exploration_candidates_average_num_rollouts: 1.5
[Step 1] Sample/mean_score: -0.0014505071706065395
[Step 1] Sample/num_samples: 2
[Step 1] Sample/self.n_epochs: 0
[Step 1] Algo/Number of training samples: 4
[Step 1] Parameter/transfer_prior:5: starting_artifact: 


### Use Case 4 — family policy & transfer (experimental)
| experiment | initial | mean score | delta | std | n | wall_s | best artifact file | notes |
|---|---:|---:|---:|---:|---:|---:|---|---|
| O2 family policy | -0.072 | 0.056 | 0.128 | 0.045 | 2 | 111.800 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc4_o2_policy_1/artifacts.jsonl#*:family_policy:0:44445` |  |
| O2->O3 (cold) | 0.007 | -0.013 | -0.020 | 0.010 | 2 | 87.700 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc4_o3_cold_0/artifacts.jsonl#*:prior:0:7085` |  |
| O2->O3 (warm prior) | 0.019 | -0.015 | -0.034 | 0.004 | 2 | 80.500 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc4_o3_warm_0/artifacts.jsonl#*:prior:0:84385` |  |


**Best: `O2 family policy`** — artifact file: `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc4_o2_policy_1/artifacts.jsonl#*:family_policy:0:44445`


gsm8k => starting_artifact= drop => starting_artifact= qasper => starting_artifact=


---
## Use Case 5 — Code helpers vs optimizer-side tools — EXPERIMENTAL

**Why:** there are two distinct meanings of “tool” here, and the notebook measures both.

**Code-helper optimization:** model a helper/selector as `CodeArtifactLevel`; the LLM rewrites
the component code and the reusable solution is saved as `kind="code"` in `artifacts.jsonl`.

**Optimizer-side tool calling:** `AgenticOptimizer` calls registered helper tools such as
`note` or `trace_search` before proposing an update, then injects their evidence into optimizer
feedback. In this notebook it does **not** learn a tool list and does **not** give downstream
agent tools to the optimized artifact.

**Mode:** offline pre-flight + LIVE for real rewrites/tool-feedback proposals. Saturated helper-code controls remain visible but are not selected as the most informative best arm.


In [8]:
# Use Case 5 — two meanings of "optimizer tool".
# 5a is CODE-SURFACE optimization of a helper/selector function. The LLM rewrites
# component code and the reusable solution is saved as kind="code" in artifacts.jsonl.
# 5b is real optimizer-side tool calling: AgenticOptimizer calls registered helper
# tools (e.g. note, trace_search) before proposing an update and injects their
# evidence into optimizer feedback. It does NOT learn tool code or a tool list here.
def _baseline_take_first(self, n, k): return list(range(k))
def _baseline_take_last(self, n, k):  return list(range(n-k, n))
def _baseline_stride(self, n, k):     return list(range(0, n, max(1, n//k)))[:k]

uc5 = []
for key, label, fn in [("take_first", "code helper: take_first", _baseline_take_first),
                       ("take_last",  "code helper: take_last",  _baseline_take_last),
                       ("stride",     "code helper: stride",     _baseline_stride)]:
    _BASELINES["batch_design"] = fn
    result = run_code_experiment("batch_design", "internal:batch_design",
                                 "Select hard/failing items first to maximize validator score.",
                                 memory_name=f"mem_uc5_code_{key}")
    if key == "stride":
        result = mark_control(result, "saturated no-op baseline: verifies persistence, not learning")
    uc5.append((label, result))
_BASELINES["batch_design"] = _weak_batch


def agentic_tool_spec(tools, label):
    spec = config_spec(
        ["starting_artifact"],
        memory_root=f"./mem_uc5_agentic_{label}",
        extra_constraints={"starting_artifact": ART_MENU},
    )
    spec["levels"] = [ make_level_spec(
        id=f"o1_agentic_{label}", surface="config", family="reasoning", task=FAMILY_TASK,
        targets=["starting_artifact"], constraints={"starting_artifact": ART_MENU},
        fixed={"optimizer": "OptoPrimeV2", "trace_type": "internal",
               "credit_horizon": "step", "trainer": "PrioritySearch"},
        agentic={"tool_budget": max(1, len(tools))}, tools=tools,
        iterations=RUN_ITERATIONS)]
    return spec

uc5 += [
    ("optimizer tools: note", run_spec_seeds(agentic_tool_spec(["note"], "note"),
                                             level_id="o1_agentic_note", run_name="mem_uc5_agentic_note")),
    ("optimizer tools: trace_search", run_spec_seeds(agentic_tool_spec(["trace_search"], "trace_search"),
                                                  level_id="o1_agentic_trace_search", run_name="mem_uc5_agentic_trace_search")),
    ("optimizer tools: trace_search+note", run_spec_seeds(agentic_tool_spec(["trace_search", "note"], "trace_note"),
                                                          level_id="o1_agentic_trace_note", run_name="mem_uc5_agentic_trace_note")),
]
show_table("Use Case 5 — helper-code vs optimizer-side tool calling", uc5)


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:00<00:00, 15534.46it/s]

Evaluating agent:   0%|          | 0/8 [00:00<?, ?it/s]

Evaluating agent: 100%|██████████| 8/8 [00:00<00:00, 4568.96it/s]

...

Evaluating agent: 100%|██████████| 2/2 [00:13<00:00,  6.55s/it]

[Step 1] Test/test_score: -0.16525
[Step 1] Algo/Average train score: -0.161625
[Step 1] Update/n_iters: 1
[Step 1] Update/short_term_memory_size: 0
[Step 1] Update/long_term_memory_size: 1
[Step 1] Update/using_short_term_memory: False
[Step 1] Update/using_long_term_memory: True
[Step 1] Update/total_samples: 2
[Step 1] Update/best_candidate_priority: -0.16049999999999998
[Step 1] Update/best_candidate_mean_score: -0.16049999999999998
[Step 1] Update/best_candidate_num_rollouts: 2
[Step 1] Update/num_exploration_candidates: 1
[Step 1] Update/exploration_candidates_mean_priority: -0.16049999999999998
[Step 1] Update/exploration_candidates_mean_score: -0.16049999999999998
[Step 1] Update/exploration_candidates_average_num_rollouts: 2.0
[Step 1] Sample/mean_score: -0.163875
[Step 1] Sample/num_samples: 1
[Step 1] Sample/self.n_epochs: 0
[Step 1] Algo/Number of training samples: 3
[Step 1] Parameter/level_config:23: starting_artifact: 


### Use Case 5 — helper-code vs optimizer-side tool calling
| experiment | initial | mean score | delta | std | n | wall_s | best artifact file | notes |
|---|---:|---:|---:|---:|---:|---:|---|---|
| code helper: take_first | 0.800 | 1.000 | 0.200 | 0.000 | 2 | 3.300 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_code_take_first_0/artifacts.jsonl#internal:batch_design:code:1:20821` |  |
| code helper: take_last | 0.700 | 1.000 | 0.300 | 0.000 | 2 | 3.500 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_code_take_last_0/artifacts.jsonl#internal:batch_design:code:1:27076` |  |
| code helper: stride | 1.000 | 1.000 | 0.000 | 0.000 | 2 | 1.600 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:31029` | saturated no-op baseline: verifies persistence, not learning |
| optimizer tools: note | -0.163 | -0.158 | 0.005 | 0.000 | 2 | 53.600 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_agentic_note_0/artifacts.jsonl#reasoning:config:0:15333` |  |
| optimizer tools: trace_search | -0.162 | -0.164 | -0.002 | 0.005 | 2 | 52.300 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_agentic_trace_search_1/artifacts.jsonl#reasoning:config:0:24025` |  |
| optimizer tools: trace_search+note | -0.164 | -0.159 | 0.005 | 0.001 | 2 | 54.200 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_agentic_trace_note_1/artifacts.jsonl#reasoning:config:0:74145` |  |


**Best: `code helper: take_last`** — artifact file: `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_code_take_last_0/artifacts.jsonl#internal:batch_design:code:1:27076`


def _baseline_take_last(self, n, k): # Prefer hard/failing indices first: indices divisible by 3. hard = [i for i in range(n) if i % 3 == 0] if len(hard) >= k: return hard[:k] # If not enough hard indices, fill remaining with the last items (excluding duplicates). remaining = [i for i in range(n - k, n) if i not in set(hard)] out = hard + remaining return out[:k]


---
## Use Case 6 — Which FEEDBACK CHANNEL helps the optimizer? (trace_type with fixed credit_horizon) — EXPERIMENTAL

**Why:** previous grids mixed too many knobs and saturated on easier tasks. This version fixes
`credit_horizon=step` from earlier evidence, then asks one controlled question: whether
`trace_type` (`internal` / `otel` / `hybrid`) changes optimizer proposals on a non-saturated
real Trace-Bench task.

The task is QASPER by default because the sampled DROP configuration saturated at 1.0 and
therefore could not distinguish trace designs. Scores are real Trace-Bench prompt/config
scores, but small-sample noise remains high.

**Mode:** needs LIVE + Trace-Bench.


In [9]:
# Use Case 6 — feedback channels. Hold credit_horizon at the strongest prior
# setting ("step") and focus on trace_type/design. This removes the noisy joint
# grid and asks one controlled question: which trace representation helps proposals?
# The full DROP run saturated at 1.0 on the sampled examples; QASPER keeps
# the trace-design comparison non-saturated while still using a real HF QA task.
UC6_TASK = HARD_PROMPT_TASKS["qasper"]

def feedback_spec(level_id, trace_type):
    return {"families": {"reasoning": [UC6_TASK]}, "memory_root": f"./mem_uc6_{level_id}",
            "budget": budget_block(), "tracebench": tracebench_block(max_examples=HARD_MAX_EXAMPLES),
            "levels": [ make_level_spec(
                id=level_id, surface="config", family="reasoning", task=UC6_TASK,
                targets=["starting_artifact"], constraints={"starting_artifact": ART_MENU},
                fixed={"optimizer": "OptoPrimeV2", "trainer": "PrioritySearch",
                       "trace_type": trace_type, "credit_horizon": "step"},
                iterations=RUN_ITERATIONS)]}

uc6 = [(f"trace_type={tt} | credit_horizon=step",
        run_spec_seeds(feedback_spec(f"o1_trace_{tt}", tt), level_id=f"o1_trace_{tt}",
                       run_name=f"mem_uc6_trace_{tt}"))
       for tt in ["internal", "otel", "hybrid"]]

show_table("Use Case 6 — trace representation with fixed step credit", uc6)


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 2 agents on 1 inputs:  50%|█████     | 1/2 [00:06<00:06,  6.72s/it]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:07<00:00,  3.31s/it]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:07<00:00,  3.82s/it]

...

Evaluating agent: 100%|██████████| 2/2 [00:07<00:00,  3.67s/it]

[Step 1] Test/test_score: 0.14231865159857046
[Step 1] Algo/Average train score: 0.13583273786158923
[Step 1] Update/n_iters: 1
[Step 1] Update/short_term_memory_size: 0
[Step 1] Update/long_term_memory_size: 3
[Step 1] Update/using_short_term_memory: False
[Step 1] Update/using_long_term_memory: True
[Step 1] Update/total_samples: 4
[Step 1] Update/best_candidate_priority: 0.14498217644508823
[Step 1] Update/best_candidate_mean_score: 0.14498217644508823
[Step 1] Update/best_candidate_num_rollouts: 2
[Step 1] Update/num_exploration_candidates: 2
[Step 1] Update/exploration_candidates_mean_priority: 0.13515369807453304
[Step 1] Update/exploration_candidates_mean_score: 0.13515369807453304
[Step 1] Update/exploration_candidates_average_num_rollouts: 1.5
[Step 1] Sample/mean_score: 0.12668329927809022
[Step 1] Sample/num_samples: 2
[Step 1] Sample/self.n_epochs: 0
[Step 1] Algo/Number of training samples: 4
[Step 1] Parameter/level_config:32: starting_artifact: 


### Use Case 6 — trace representation with fixed step credit
| experiment | initial | mean score | delta | std | n | wall_s | best artifact file | notes |
|---|---:|---:|---:|---:|---:|---:|---|---|
| trace_type=internal \| credit_horizon=step | 0.161 | 0.181 | 0.021 | 0.013 | 2 | 36.600 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc6_trace_internal_1/artifacts.jsonl#reasoning:config:0:69377` |  |
| trace_type=otel \| credit_horizon=step | 0.117 | 0.187 | 0.070 | 0.066 | 2 | 42.200 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc6_trace_otel_0/artifacts.jsonl#reasoning:config:0:26419` |  |
| trace_type=hybrid \| credit_horizon=step | 0.121 | 0.161 | 0.040 | 0.027 | 2 | 36.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc6_trace_hybrid_0/artifacts.jsonl#reasoning:config:0:24828` |  |


**Best: `trace_type=otel \| credit_horizon=step`** — artifact file: `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc6_trace_otel_0/artifacts.jsonl#reasoning:config:0:26419`


starting_artifact:


---
## Master summary — all use cases at a glance

Run after the experiments above. The first table shows **every current-run experiment** with
initial score, mean score, delta, wall time, and the file/id of the best saved artifact. The
second table picks one best non-control row per use case; interpret saturated rows with the guardrails below.
The historical tables scan all persisted `examples/notebook_outputs/recursive_opt_use_cases`
runs so previous artifacts can be compared and reused.


## Use Case 7 — Graph routing to a sub-optimizer tool — PROBE

**Why:** this isolates the “use another optimizer as a tool/sub-optimizer” question from Trace-Bench noise. The graph starts with a weak draft route (`route_policy=draft`) and has a deterministic SciPy sub-optimizer node available. The optimizer only learns the graph routing/design knob, not the SciPy code itself.

**Mode:** needs LIVE because the graph route is selected by the LLM optimizer. The output artifact stores the learned graph parameter (`route_policy=scipy`) plus score history.


In [ ]:
# Use Case 7 — graph routing to a downstream sub-optimizer tool.
# This is the missing C-style probe: learn when to route work to a real local
# optimizer instead of keeping a weak draft answer. It is intentionally small and
# deterministic so the graph/control-surface effect is easy to interpret.
from argparse import Namespace
from examples.recursive_opt_abc_probe import run_suboptimizer_graph


def run_suboptimizer_use_case():
    """Run the graph/suboptimizer probe and return a table-compatible result."""
    if not LIVE:
        return {"scores": [], "initial": None, "wall_s": None,
                "artifact": "(dry-run: set LIVE=True to optimize graph route)",
                "artifact_id": None, "artifact_file": None, "dry": True}
    reset_standard_budget()
    args = Namespace(model=MODEL, iterations=RUN_ITERATIONS, candidates=NUM_CANDIDATES,
                     max_examples=MAX_EXAMPLES, timeout_seconds=TIMEOUT_S,
                     live=True, skip_preflight=True)
    result = run_suboptimizer_graph(OUTPUT_ROOT, args)
    artifact = json.dumps({
        "params": result.get("params"),
        "score_history": result.get("score_history"),
        "oracle_tool_score": result.get("oracle_tool_score"),
    }, indent=2, sort_keys=True)
    return {
        "scores": [float(result["final"])],
        "initial": float(result["initial"]),
        "wall_s": float(result["wall_s"]),
        "artifact": artifact,
        "artifact_id": "graph:suboptimizer:latest",
        "artifact_file": result.get("artifact_file"),
        "errors": [],
        "control_reason": "learned graph route to SciPy sub-optimizer",
    }


uc7 = [("graph route: SciPy suboptimizer tool", run_suboptimizer_use_case())]
show_table("Use Case 7 — graph/suboptimizer routing", uc7)


### Use Case 7 — graph/suboptimizer routing

| experiment | initial | mean score | delta | std | n | wall_s | best artifact file | notes |
|---|---:|---:|---:|---:|---:|---:|---|---|
| graph route: SciPy suboptimizer tool | 0.000 | 1.000 | 1.000 | - | 1 | 4.629 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_suboptimizer_graph/artifacts.jsonl#graph:suboptimizer:latest` | learned graph route to SciPy sub-optimizer |


**Best: `graph route: SciPy suboptimizer tool`** — artifact file: `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_suboptimizer_graph/artifacts.jsonl#graph:suboptimizer:latest`


{
  "oracle_tool_score": 1.0,
  "params": {
    "route_policy": "scipy"
  },
  "score_history": [
    0.0,
    0.0,
    1.0
  ]
}


In [10]:
# Master roll-up: all experiments, best per use case, and previous-run artifact summaries.
ALL = {"UC1 component code": uc1, "UC2 setup/config": uc2, "UC3 capability": uc3,
       "UC4 family/transfer": uc4, "UC5 optimizer/tool": uc5, "UC6 trace feedback": uc6,
       "UC7 graph/suboptimizer": uc7}

flat = ["| use case | experiment | initial | mean score | delta | std | n | wall_s | best artifact file | notes | best? |",
        "|---|---|---:|---:|---:|---:|---:|---:|---|---|---|"]
for uc, data in ALL.items():
    best = best_of(data)
    best_label = best[0] if best else None
    for label, result in data:
        scores = _finite(result.get("scores", []))
        mean = statistics.mean(scores) if scores else None
        std = statistics.pstdev(scores) if len(scores) > 1 else None
        delta = (mean - result["initial"]) if mean is not None and result.get("initial") is not None else None
        flat.append(f"| {_md_cell(uc)} | {_md_cell(label)} | {_fmt(result.get('initial'))} | {_fmt(mean)} | {_fmt(delta)} | "
                    f"{_fmt(std)} | {len(scores)} | {_fmt(result.get('wall_s'))} | "
                    f"{_md_code(result.get('artifact_file') or '-')} | {_md_cell(_notes_for_result(result))} | "
                    f"{'yes' if label == best_label else ''} |")

_display_markdown("### All current-run results\n" + "\n".join(flat))

best_rows = ["| use case | best experiment | initial | mean score | delta | n | wall_s | best artifact file |",
             "|---|---|---:|---:|---:|---:|---:|---|"]
for uc, data in ALL.items():
    b = best_of(data)
    if b is None:
        best_rows.append(f"| {_md_cell(uc)} | (dry-run / no live result) | - | - | - | 0 | - | - |")
    else:
        label, result = b
        mean = _result_mean(result)
        delta = (mean - result["initial"]) if mean is not None and result.get("initial") is not None else None
        best_rows.append(f"| {_md_cell(uc)} | {_md_cell(label)} | {_fmt(result.get('initial'))} | {_fmt(mean)} | {_fmt(delta)} | "
                         f"{len(_finite(result.get('scores', [])))} | {_fmt(result.get('wall_s'))} | "
                         f"{_md_code(result.get('artifact_file') or '-')} |")
_display_markdown("### Best result per use case\n" + "\n".join(best_rows))

past = summarize_past_runs(OUTPUT_ROOT.parent)
if past:
    _display_markdown("### Historical persisted-artifact summary\n" + past_runs_table(past))
    detailed = summarize_past_experiments(OUTPUT_ROOT.parent)
    _display_markdown("### Historical persisted-artifact detail (all past experiments)\n" + past_experiments_table(detailed))
else:
    _display_markdown("### Historical persisted-artifact summary\nNo prior output folders found.")

_display_markdown("**Interpretation guardrails:** UC1/UC5 code-helper scores are validator evidence and can saturate; "
                 "UC2/UC6 are real Trace-Bench prompt/config scores over the configured examples; "
                 "UC1 now includes a real BBEH direct-code arm to avoid relying only on toy validators; "
                 "UC3 remains a prompt-capability surface on GSM8K; UC4 transfer is only meaningful when warm beats cold by more than run noise. "
                 "For config arms, inspect whether the saved config actually changed; if `starting_artifact` is blank/unchanged, treat the gain as benchmark/trace variance or trace-condition evidence, not as a learned prompt. "
                 "Reusable artifacts are in each listed `artifacts.jsonl` under the `content` field; code arms save Python code, config arms save config text, and optimizer-tool arms save tool names/evidence hooks.")


### All current-run results
| use case | experiment | initial | mean score | delta | std | n | wall_s | best artifact file | notes | best? |
|---|---|---:|---:|---:|---:|---:|---:|---|---|---|
| UC1 component code | batch_design (failure-balanced) | 0.800 | 1.000 | 0.200 | 0.000 | 2 | 3.500 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:30451` |  |  |
| UC1 component code | trace_summarizer (default) | 0.750 | 0.897 | 0.147 | 0.021 | 2 | 3.200 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc1_trace_summarizer_default_1/artifacts.jsonl#internal:code_param:code:1:40708` |  |  |
| UC1 component code | trace_summarizer (strict) | 0.750 | 0.897 | 0.147 | 0.021 | 2 | 4.100 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc1_trace_summarizer_strict_0/artifacts.jsonl#internal:code_param:code:1:44914` |  |  |
| UC1 component code | BBEH direct code solver (real hard examples) | 0.625 | 1.000 | 0.375 | 0.000 | 2 | 4.900 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc1_bbeh_direct_solver_0/artifacts.jsonl#internal:multiobjective_bbeh:code:1:54006` |  | yes |
| UC2 setup/config | artifact only | -0.163 | -0.136 | 0.026 | 0.011 | 2 | 66.500 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_artifact_only_1/artifacts.jsonl#reasoning:config:0:30223` |  |  |
| UC2 setup/config | artifact+knowledge | -0.156 | -0.161 | -0.005 | 0.004 | 2 | 65.600 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_artifact_knowledge_1/artifacts.jsonl#reasoning:config:0:719` |  |  |
| UC2 setup/config | artifact (warm prior) | -0.162 | -0.134 | 0.028 | 0.018 | 2 | 65.100 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_warm_prior_0/artifacts.jsonl#reasoning:config:0:87661` |  |  |
| UC2 setup/config | artifact on DROP (QA control; often saturated) | 0.750 | 0.875 | 0.125 | 0.125 | 2 | 36.300 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_drop_1/artifacts.jsonl#drop:config:0:60919` | saturated control: useful for comparison, not selected as best |  |
| UC2 setup/config | artifact on QASPER (harder QA) | 0.113 | 0.163 | 0.049 | 0.020 | 2 | 39.900 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_qasper_1/artifacts.jsonl#qasper:config:0:62209` |  | yes |
| UC3 capability | seed: terse | 0.968 | 0.878 | -0.090 | 0.090 | 2 | 53.200 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc3_terse_0/artifacts.jsonl#reasoning:capability:0:38889` |  |  |
| UC3 capability | seed: verify | 1.441 | 1.378 | -0.062 | 0.062 | 2 | 63.600 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc3_verify_0/artifacts.jsonl#reasoning:capability:0:96998` |  |  |
| UC3 capability | seed: decompose | 1.439 | 1.445 | 0.006 | 0.006 | 2 | 63.700 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc3_decompose_0/artifacts.jsonl#reasoning:capability:0:59196` |  | yes |
| UC4 family/transfer | O2 family policy | -0.072 | 0.056 | 0.128 | 0.045 | 2 | 111.800 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc4_o2_policy_1/artifacts.jsonl#*:family_policy:0:44445` |  | yes |
| UC4 family/transfer | O2->O3 (cold) | 0.007 | -0.013 | -0.020 | 0.010 | 2 | 87.700 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc4_o3_cold_0/artifacts.jsonl#*:prior:0:7085` |  |  |
| UC4 family/transfer | O2->O3 (warm prior) | 0.019 | -0.015 | -0.034 | 0.004 | 2 | 80.500 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc4_o3_warm_0/artifacts.jsonl#*:prior:0:84385` |  |  |
| UC5 optimizer/tool | code helper: take_first | 0.800 | 1.000 | 0.200 | 0.000 | 2 | 3.300 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_code_take_first_0/artifacts.jsonl#internal:batch_design:code:1:20821` |  |  |
| UC5 optimizer/tool | code helper: take_last | 0.700 | 1.000 | 0.300 | 0.000 | 2 | 3.500 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_code_take_last_0/artifacts.jsonl#internal:batch_design:code:1:27076` |  | yes |
| UC5 optimizer/tool | code helper: stride | 1.000 | 1.000 | 0.000 | 0.000 | 2 | 1.600 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:31029` | saturated no-op baseline: verifies persistence, not learning |  |
| UC5 optimizer/tool | optimizer tools: note | -0.163 | -0.158 | 0.005 | 0.000 | 2 | 53.600 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_agentic_note_0/artifacts.jsonl#reasoning:config:0:15333` |  |  |
| UC5 optimizer/tool | optimizer tools: trace_search | -0.162 | -0.164 | -0.002 | 0.005 | 2 | 52.300 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_agentic_trace_search_1/artifacts.jsonl#reasoning:config:0:24025` |  |  |
| UC5 optimizer/tool | optimizer tools: trace_search+note | -0.164 | -0.159 | 0.005 | 0.001 | 2 | 54.200 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_agentic_trace_note_1/artifacts.jsonl#reasoning:config:0:74145` |  |  |
| UC6 trace feedback | trace_type=internal \| credit_horizon=step | 0.161 | 0.181 | 0.021 | 0.013 | 2 | 36.600 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc6_trace_internal_1/artifacts.jsonl#reasoning:config:0:69377` |  |  |
| UC6 trace feedback | trace_type=otel \| credit_horizon=step | 0.117 | 0.187 | 0.070 | 0.066 | 2 | 42.200 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc6_trace_otel_0/artifacts.jsonl#reasoning:config:0:26419` |  | yes |
| UC6 trace feedback | trace_type=hybrid \| credit_horizon=step | 0.121 | 0.161 | 0.040 | 0.027 | 2 | 36.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc6_trace_hybrid_0/artifacts.jsonl#reasoning:config:0:24828` |  |  |
| UC7 graph/suboptimizer | graph route: SciPy suboptimizer tool | 0.000 | 1.000 | 1.000 | - | 1 | 4.629 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_suboptimizer_graph/artifacts.jsonl#graph:suboptimizer:latest` | learned graph route to SciPy sub-optimizer | yes |


### Best result per use case
| use case | best experiment | initial | mean score | delta | n | wall_s | best artifact file |
|---|---|---:|---:|---:|---:|---:|---|
| UC1 component code | BBEH direct code solver (real hard examples) | 0.625 | 1.000 | 0.375 | 2 | 4.900 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc1_bbeh_direct_solver_0/artifacts.jsonl#internal:multiobjective_bbeh:code:1:54006` |
| UC2 setup/config | artifact on QASPER (harder QA) | 0.113 | 0.163 | 0.049 | 2 | 39.900 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_qasper_1/artifacts.jsonl#qasper:config:0:62209` |
| UC3 capability | seed: decompose | 1.439 | 1.445 | 0.006 | 2 | 63.700 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc3_decompose_0/artifacts.jsonl#reasoning:capability:0:59196` |
| UC4 family/transfer | O2 family policy | -0.072 | 0.056 | 0.128 | 2 | 111.800 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc4_o2_policy_1/artifacts.jsonl#*:family_policy:0:44445` |
| UC5 optimizer/tool | code helper: take_last | 0.700 | 1.000 | 0.300 | 2 | 3.500 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_code_take_last_0/artifacts.jsonl#internal:batch_design:code:1:27076` |
| UC6 trace feedback | trace_type=otel \| credit_horizon=step | 0.117 | 0.187 | 0.070 | 2 | 42.200 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc6_trace_otel_0/artifacts.jsonl#reasoning:config:0:26419` |
| UC7 graph/suboptimizer | graph route: SciPy suboptimizer tool | 0.000 | 1.000 | 1.000 | 1 | 4.629 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_suboptimizer_graph/artifacts.jsonl#graph:suboptimizer:latest` |


### Historical persisted-artifact summary
| run | use case | initial mean | final mean | best score | n dirs | best artifact file |
|---|---|---:|---:|---:|---:|---|
| use_cases_live_20260613_215505 | UC1 component code | 0.775 | 0.936 | 1.000 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:0:13999` |
| use_cases_live_20260613_215505 | UC3 capability | 0.964 | 0.964 | 0.968 | 3 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc3_0/artifacts.jsonl#reasoning:capability:0:94067` |
| use_cases_live_20260613_215505 | UC4 family/transfer | -0.579 | -0.366 | -0.153 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc4_o3_0/artifacts.jsonl#<holdout>:prior:0:93503` |
| use_cases_live_20260613_215505 | UC5 optimizer/tool | 0.833 | 1.000 | 1.000 | 9 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc5_stride_0/artifacts.jsonl#internal:batch_design:code:0:28880` |
| use_cases_live_deep_20260614_000827 | UC1 component code | 0.775 | 0.939 | 1.000 | 4 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:48142` |
| use_cases_live_deep_20260614_000827 | UC2 setup/config | -0.149 | -0.149 | -0.143 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc2_artifact_only_1/artifacts.jsonl#reasoning:config:0:18120` |
| use_cases_live_deep_20260614_000827 | UC4 family/transfer | 0.419 | 0.807 | 1.000 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc4_o3_warm_0/artifacts.jsonl#<holdout>:prior:0:3687` |
| use_cases_live_deep_20260614_000827 | UC5 optimizer/tool | 0.434 | 0.534 | 1.000 | 10 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:84588` |
| use_cases_live_deep_20260614_000827 | UC6 trace feedback | -0.150 | -0.150 | -0.144 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc6_trace_otel_0/artifacts.jsonl#reasoning:config:0:61144` |
| use_cases_live_deep_budgeted_20260614_012157 | UC1 component code | 0.767 | 0.882 | 1.000 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:71116` |
| use_cases_live_deep_budgeted_20260614_012157 | UC2 setup/config | -0.148 | -0.148 | -0.140 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc2_artifact_only_1/artifacts.jsonl#reasoning:config:0:38164` |
| use_cases_live_deep_budgeted_20260614_012157 | UC3 capability | 1.262 | 1.262 | 1.441 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc3_verify_0/artifacts.jsonl#reasoning:capability:0:31701` |
| use_cases_live_deep_budgeted_20260614_012157 | UC4 family/transfer | 0.419 | 0.807 | 1.000 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc4_o3_cold_0/artifacts.jsonl#<holdout>:prior:0:24095` |
| use_cases_live_deep_budgeted_20260614_012157 | UC5 optimizer/tool | 0.435 | 0.535 | 1.000 | 10 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:25130` |
| use_cases_live_deep_budgeted_20260614_012157 | UC6 trace feedback | -0.136 | -0.136 | -0.118 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc6_trace_internal_1/artifacts.jsonl#reasoning:config:0:3911` |
| use_cases_live_deep_fixed_20260614_004207 | UC1 component code | 0.767 | 0.890 | 1.000 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:70343` |
| use_cases_live_deep_fixed_20260614_004207 | UC2 setup/config | -0.151 | -0.151 | -0.143 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc2_artifact_only_1/artifacts.jsonl#reasoning:config:0:34862` |
| use_cases_live_deep_fixed_20260614_004207 | UC4 family/transfer | 0.419 | 0.807 | 1.000 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc4_o3_cold_0/artifacts.jsonl#<holdout>:prior:0:61920` |
| use_cases_live_deep_fixed_20260614_004207 | UC5 optimizer/tool | 0.435 | 0.535 | 1.000 | 10 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:74840` |
| use_cases_live_deep_fixed_20260614_004207 | UC6 trace feedback | -0.139 | -0.139 | -0.116 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc6_trace_hybrid_0/artifacts.jsonl#reasoning:config:0:46689` |
| use_cases_live_fixed_20260613_221437 | UC1 component code | 0.775 | 0.960 | 1.000 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:0:85257` |
| use_cases_live_fixed_20260613_221437 | UC2 setup/config | -0.143 | -0.132 | -0.116 | 3 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc2_1/artifacts.jsonl#reasoning:config:2:64082` |
| use_cases_live_fixed_20260613_221437 | UC3 capability | 0.966 | 0.966 | 0.968 | 3 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc3_0/artifacts.jsonl#reasoning:capability:0:31303` |
| use_cases_live_fixed_20260613_221437 | UC4 family/transfer | 0.421 | 0.711 | 1.000 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc4_o3_2/artifacts.jsonl#<holdout>:prior:0:31092` |
| use_cases_live_fixed_20260613_221437 | UC5 optimizer/tool | 0.833 | 1.000 | 1.000 | 9 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc5_stride_0/artifacts.jsonl#internal:batch_design:code:0:86973` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | -0.144 | -0.144 | -0.120 | 21 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_ch_step_0/artifacts.jsonl#reasoning:config:0:28025` |
| use_cases_rootcause_20260614_022600 | UC1 component code | 0.731 | 0.925 | 1.000 | 8 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:99106` |
| use_cases_rootcause_20260614_022600 | UC2 setup/config | 0.148 | 0.148 | 1.000 | 10 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc2_drop_0/artifacts.jsonl#drop:config:0:81549` |
| use_cases_rootcause_20260614_022600 | UC3 capability | 1.319 | 1.319 | 1.441 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc3_verify_0/artifacts.jsonl#reasoning:capability:0:32066` |
| use_cases_rootcause_20260614_022600 | UC4 family/transfer | 0.055 | 0.075 | 0.129 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc4_o3_cold_0/artifacts.jsonl#<multi>:policy:0:56150` |
| use_cases_rootcause_20260614_022600 | UC5 optimizer/tool | 0.434 | 0.534 | 1.000 | 10 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:22248` |
| use_cases_rootcause_20260614_022600 | UC6 trace feedback | 0.567 | 0.567 | 1.000 | 12 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc6_trace_hybrid_0/artifacts.jsonl#reasoning:config:0:24217` |
| use_cases_rootcause_final_20260614 | UC1 component code | 0.731 | 0.948 | 1.000 | 8 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:30451` |
| use_cases_rootcause_final_20260614 | UC2 setup/config | 0.121 | 0.121 | 1.000 | 10 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_drop_1/artifacts.jsonl#drop:config:0:60919` |
| use_cases_rootcause_final_20260614 | UC3 capability | 1.234 | 1.234 | 1.451 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc3_decompose_0/artifacts.jsonl#reasoning:capability:0:59196` |
| use_cases_rootcause_final_20260614 | UC4 family/transfer | 0.043 | 0.062 | 0.101 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc4_o2_policy_1/artifacts.jsonl#<multi>:policy:0:44445` |
| use_cases_rootcause_final_20260614 | UC5 optimizer/tool | 0.336 | 0.420 | 1.000 | 12 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:31029` |
| use_cases_rootcause_final_20260614 | UC6 trace feedback | 0.177 | 0.177 | 0.253 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc6_trace_otel_0/artifacts.jsonl#reasoning:config:0:26419` |
| use_cases_rootcause_final_20260614 | UC7 graph/suboptimizer | 0.000 | 1.000 | 1.000 | 1 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_suboptimizer_graph/artifacts.jsonl#graph:suboptimizer:latest` |
| use_cases_rootcause_qasper_20260614_032217 | UC1 component code | 0.731 | 0.893 | 1.000 | 8 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:65938` |
| use_cases_rootcause_qasper_20260614_032217 | UC2 setup/config | 0.141 | 0.141 | 1.000 | 10 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc2_drop_0/artifacts.jsonl#drop:config:0:49092` |
| use_cases_rootcause_qasper_20260614_032217 | UC3 capability | 1.263 | 1.263 | 1.448 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc3_decompose_0/artifacts.jsonl#reasoning:capability:0:54745` |
| use_cases_rootcause_qasper_20260614_032217 | UC4 family/transfer | -0.051 | 0.060 | 0.158 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc4_o3_cold_1/artifacts.jsonl#<holdout>:prior:0:5603` |
| use_cases_rootcause_qasper_20260614_032217 | UC5 optimizer/tool | 0.435 | 0.535 | 1.000 | 10 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:79161` |
| use_cases_rootcause_qasper_20260614_032217 | UC6 trace feedback | 0.155 | 0.155 | 0.201 | 6 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc6_trace_internal_0/artifacts.jsonl#reasoning:config:0:28292` |


### Historical persisted-artifact detail (all past experiments)
| run | use case | experiment | initial | best score | best artifact file |
|---|---|---|---:|---:|---|
| use_cases_live_20260613_215505 | UC1 component code | mem_uc1_batch_design | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:0:13999` |
| use_cases_live_20260613_215505 | UC1 component code | mem_uc1_batch_design | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc1_batch_design_1/artifacts.jsonl#internal:batch_design:code:0:17501` |
| use_cases_live_20260613_215505 | UC1 component code | mem_uc1_batch_design | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc1_batch_design_2/artifacts.jsonl#internal:batch_design:code:0:21503` |
| use_cases_live_20260613_215505 | UC1 component code | mem_uc1_trace_summarizer | 0.750 | 0.778 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc1_trace_summarizer_0/artifacts.jsonl#internal:code_param:code:14:37065` |
| use_cases_live_20260613_215505 | UC1 component code | mem_uc1_trace_summarizer | 0.750 | 0.876 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc1_trace_summarizer_1/artifacts.jsonl#internal:code_param:code:15:39982` |
| use_cases_live_20260613_215505 | UC1 component code | mem_uc1_trace_summarizer | 0.750 | 0.959 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc1_trace_summarizer_2/artifacts.jsonl#internal:code_param:code:15:43258` |
| use_cases_live_20260613_215505 | UC3 capability | mem_uc3 | 0.968 | 0.968 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc3_0/artifacts.jsonl#reasoning:capability:0:94067` |
| use_cases_live_20260613_215505 | UC3 capability | mem_uc3 | 0.962 | 0.962 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc3_1/artifacts.jsonl#reasoning:capability:0:43059` |
| use_cases_live_20260613_215505 | UC3 capability | mem_uc3 | 0.962 | 0.962 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc3_2/artifacts.jsonl#reasoning:capability:0:64875` |
| use_cases_live_20260613_215505 | UC4 family/transfer | mem_uc4_o2 | -0.580 | -0.577 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc4_o2_0/artifacts.jsonl#<multi>:policy:0:71629` |
| use_cases_live_20260613_215505 | UC4 family/transfer | mem_uc4_o2 | -0.579 | -0.578 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc4_o2_1/artifacts.jsonl#<multi>:policy:0:17712` |
| use_cases_live_20260613_215505 | UC4 family/transfer | mem_uc4_o2 | -0.578 | -0.578 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc4_o2_2/artifacts.jsonl#<multi>:policy:0:29384` |
| use_cases_live_20260613_215505 | UC4 family/transfer | mem_uc4_o3 | -0.578 | -0.153 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc4_o3_0/artifacts.jsonl#<holdout>:prior:0:93503` |
| use_cases_live_20260613_215505 | UC4 family/transfer | mem_uc4_o3 | -0.577 | -0.154 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc4_o3_1/artifacts.jsonl#<holdout>:prior:11:84773` |
| use_cases_live_20260613_215505 | UC4 family/transfer | mem_uc4_o3 | -0.579 | -0.156 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc4_o3_2/artifacts.jsonl#<holdout>:prior:0:40342` |
| use_cases_live_20260613_215505 | UC5 optimizer/tool | mem_uc5_stride | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc5_stride_0/artifacts.jsonl#internal:batch_design:code:0:28880` |
| use_cases_live_20260613_215505 | UC5 optimizer/tool | mem_uc5_stride | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc5_stride_1/artifacts.jsonl#internal:batch_design:code:0:29869` |
| use_cases_live_20260613_215505 | UC5 optimizer/tool | mem_uc5_stride | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc5_stride_2/artifacts.jsonl#internal:batch_design:code:0:30501` |
| use_cases_live_20260613_215505 | UC5 optimizer/tool | mem_uc5_take_first | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc5_take_first_0/artifacts.jsonl#internal:batch_design:code:0:11823` |
| use_cases_live_20260613_215505 | UC5 optimizer/tool | mem_uc5_take_first | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc5_take_first_1/artifacts.jsonl#internal:batch_design:code:0:15142` |
| use_cases_live_20260613_215505 | UC5 optimizer/tool | mem_uc5_take_first | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc5_take_first_2/artifacts.jsonl#internal:batch_design:code:0:18408` |
| use_cases_live_20260613_215505 | UC5 optimizer/tool | mem_uc5_take_last | 0.700 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc5_take_last_0/artifacts.jsonl#internal:batch_design:code:0:21660` |
| use_cases_live_20260613_215505 | UC5 optimizer/tool | mem_uc5_take_last | 0.700 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc5_take_last_1/artifacts.jsonl#internal:batch_design:code:0:25105` |
| use_cases_live_20260613_215505 | UC5 optimizer/tool | mem_uc5_take_last | 0.700 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_20260613_215505/mem_uc5_take_last_2/artifacts.jsonl#internal:batch_design:code:0:28845` |
| use_cases_live_deep_20260614_000827 | UC1 component code | mem_uc1_batch_design | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:48142` |
| use_cases_live_deep_20260614_000827 | UC1 component code | mem_uc1_batch_design | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc1_batch_design_1/artifacts.jsonl#internal:batch_design:code:1:50819` |
| use_cases_live_deep_20260614_000827 | UC1 component code | mem_uc1_trace_summarizer | 0.750 | 0.834 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc1_trace_summarizer_0/artifacts.jsonl#internal:code_param:code:1:53877` |
| use_cases_live_deep_20260614_000827 | UC1 component code | mem_uc1_trace_summarizer | 0.750 | 0.923 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc1_trace_summarizer_1/artifacts.jsonl#internal:code_param:code:25:61712` |
| use_cases_live_deep_20260614_000827 | UC2 setup/config | mem_uc2_artifact_knowledge | -0.151 | -0.151 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc2_artifact_knowledge_0/artifacts.jsonl#reasoning:config:0:4958` |
| use_cases_live_deep_20260614_000827 | UC2 setup/config | mem_uc2_artifact_knowledge | -0.165 | -0.165 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc2_artifact_knowledge_1/artifacts.jsonl#reasoning:config:0:74376` |
| use_cases_live_deep_20260614_000827 | UC2 setup/config | mem_uc2_artifact_only | -0.148 | -0.148 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc2_artifact_only_0/artifacts.jsonl#reasoning:config:0:47968` |
| use_cases_live_deep_20260614_000827 | UC2 setup/config | mem_uc2_artifact_only | -0.143 | -0.143 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc2_artifact_only_1/artifacts.jsonl#reasoning:config:0:18120` |
| use_cases_live_deep_20260614_000827 | UC2 setup/config | mem_uc2_warm_prior | -0.144 | -0.144 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc2_warm_prior_0/artifacts.jsonl#reasoning:config:0:53318` |
| use_cases_live_deep_20260614_000827 | UC2 setup/config | mem_uc2_warm_prior | -0.145 | -0.145 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc2_warm_prior_1/artifacts.jsonl#reasoning:config:0:24904` |
| use_cases_live_deep_20260614_000827 | UC4 family/transfer | mem_uc4_o2_policy | 0.419 | 0.421 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc4_o2_policy_0/artifacts.jsonl#<multi>:policy:0:36002` |
| use_cases_live_deep_20260614_000827 | UC4 family/transfer | mem_uc4_o2_policy | 0.419 | 0.419 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc4_o2_policy_1/artifacts.jsonl#<multi>:policy:0:58224` |
| use_cases_live_deep_20260614_000827 | UC4 family/transfer | mem_uc4_o3_cold | 0.419 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc4_o3_cold_0/artifacts.jsonl#<holdout>:prior:0:67275` |
| use_cases_live_deep_20260614_000827 | UC4 family/transfer | mem_uc4_o3_cold | 0.419 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc4_o3_cold_1/artifacts.jsonl#<holdout>:prior:0:38582` |
| use_cases_live_deep_20260614_000827 | UC4 family/transfer | mem_uc4_o3_warm | 0.421 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc4_o3_warm_0/artifacts.jsonl#<holdout>:prior:0:3687` |
| use_cases_live_deep_20260614_000827 | UC4 family/transfer | mem_uc4_o3_warm | 0.418 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc4_o3_warm_1/artifacts.jsonl#<holdout>:prior:0:66113` |
| use_cases_live_deep_20260614_000827 | UC5 optimizer/tool | mem_uc5_agentic_note | -0.173 | -0.173 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc5_agentic_note_0/artifacts.jsonl#reasoning:config:0:59015` |
| use_cases_live_deep_20260614_000827 | UC5 optimizer/tool | mem_uc5_agentic_note | -0.160 | -0.160 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc5_agentic_note_1/artifacts.jsonl#reasoning:config:0:12786` |
| use_cases_live_deep_20260614_000827 | UC5 optimizer/tool | mem_uc5_agentic_trace_note | -0.161 | -0.161 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc5_agentic_trace_note_0/artifacts.jsonl#reasoning:config:0:77084` |
| use_cases_live_deep_20260614_000827 | UC5 optimizer/tool | mem_uc5_agentic_trace_note | -0.164 | -0.164 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc5_agentic_trace_note_1/artifacts.jsonl#reasoning:config:0:32032` |
| use_cases_live_deep_20260614_000827 | UC5 optimizer/tool | mem_uc5_code_stride | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:84588` |
| use_cases_live_deep_20260614_000827 | UC5 optimizer/tool | mem_uc5_code_stride | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc5_code_stride_1/artifacts.jsonl#internal:batch_design:code:0:87307` |
| use_cases_live_deep_20260614_000827 | UC5 optimizer/tool | mem_uc5_code_take_first | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc5_code_take_first_0/artifacts.jsonl#internal:batch_design:code:1:73528` |
| use_cases_live_deep_20260614_000827 | UC5 optimizer/tool | mem_uc5_code_take_first | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc5_code_take_first_1/artifacts.jsonl#internal:batch_design:code:1:76829` |
| use_cases_live_deep_20260614_000827 | UC5 optimizer/tool | mem_uc5_code_take_last | 0.700 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc5_code_take_last_0/artifacts.jsonl#internal:batch_design:code:1:80208` |
| use_cases_live_deep_20260614_000827 | UC5 optimizer/tool | mem_uc5_code_take_last | 0.700 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc5_code_take_last_1/artifacts.jsonl#internal:batch_design:code:1:84503` |
| use_cases_live_deep_20260614_000827 | UC6 trace feedback | mem_uc6_trace_hybrid | -0.145 | -0.145 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc6_trace_hybrid_0/artifacts.jsonl#reasoning:config:0:12499` |
| use_cases_live_deep_20260614_000827 | UC6 trace feedback | mem_uc6_trace_hybrid | -0.146 | -0.146 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc6_trace_hybrid_1/artifacts.jsonl#reasoning:config:0:17764` |
| use_cases_live_deep_20260614_000827 | UC6 trace feedback | mem_uc6_trace_internal | -0.153 | -0.153 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc6_trace_internal_0/artifacts.jsonl#reasoning:config:0:14708` |
| use_cases_live_deep_20260614_000827 | UC6 trace feedback | mem_uc6_trace_internal | -0.160 | -0.160 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc6_trace_internal_1/artifacts.jsonl#reasoning:config:0:80453` |
| use_cases_live_deep_20260614_000827 | UC6 trace feedback | mem_uc6_trace_otel | -0.144 | -0.144 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc6_trace_otel_0/artifacts.jsonl#reasoning:config:0:61144` |
| use_cases_live_deep_20260614_000827 | UC6 trace feedback | mem_uc6_trace_otel | -0.152 | -0.152 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_20260614_000827/mem_uc6_trace_otel_1/artifacts.jsonl#reasoning:config:0:30266` |
| use_cases_live_deep_budgeted_20260614_012157 | UC1 component code | mem_uc1_batch_design | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:71116` |
| use_cases_live_deep_budgeted_20260614_012157 | UC1 component code | mem_uc1_batch_design | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc1_batch_design_1/artifacts.jsonl#internal:batch_design:code:1:75275` |
| use_cases_live_deep_budgeted_20260614_012157 | UC1 component code | mem_uc1_trace_summarizer_default | 0.750 | 0.750 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc1_trace_summarizer_default_0/artifacts.jsonl#internal:code_param:code:0:75412` |
| use_cases_live_deep_budgeted_20260614_012157 | UC1 component code | mem_uc1_trace_summarizer_default | 0.750 | 0.876 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc1_trace_summarizer_default_1/artifacts.jsonl#internal:code_param:code:1:82514` |
| use_cases_live_deep_budgeted_20260614_012157 | UC1 component code | mem_uc1_trace_summarizer_strict | 0.750 | 0.750 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc1_trace_summarizer_strict_0/artifacts.jsonl#internal:code_param:code:0:82639` |
| use_cases_live_deep_budgeted_20260614_012157 | UC1 component code | mem_uc1_trace_summarizer_strict | 0.750 | 0.917 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc1_trace_summarizer_strict_1/artifacts.jsonl#internal:code_param:code:1:89689` |
| use_cases_live_deep_budgeted_20260614_012157 | UC2 setup/config | mem_uc2_artifact_knowledge | -0.150 | -0.150 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc2_artifact_knowledge_0/artifacts.jsonl#reasoning:config:0:21713` |
| use_cases_live_deep_budgeted_20260614_012157 | UC2 setup/config | mem_uc2_artifact_knowledge | -0.156 | -0.156 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc2_artifact_knowledge_1/artifacts.jsonl#reasoning:config:0:92626` |
| use_cases_live_deep_budgeted_20260614_012157 | UC2 setup/config | mem_uc2_artifact_only | -0.147 | -0.147 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc2_artifact_only_0/artifacts.jsonl#reasoning:config:0:68605` |
| use_cases_live_deep_budgeted_20260614_012157 | UC2 setup/config | mem_uc2_artifact_only | -0.140 | -0.140 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc2_artifact_only_1/artifacts.jsonl#reasoning:config:0:38164` |
| use_cases_live_deep_budgeted_20260614_012157 | UC2 setup/config | mem_uc2_warm_prior | -0.142 | -0.142 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc2_warm_prior_0/artifacts.jsonl#reasoning:config:0:72083` |
| use_cases_live_deep_budgeted_20260614_012157 | UC2 setup/config | mem_uc2_warm_prior | -0.153 | -0.153 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc2_warm_prior_1/artifacts.jsonl#reasoning:config:0:42157` |
| use_cases_live_deep_budgeted_20260614_012157 | UC3 capability | mem_uc3_decompose | 1.439 | 1.439 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc3_decompose_0/artifacts.jsonl#reasoning:capability:0:78235` |
| use_cases_live_deep_budgeted_20260614_012157 | UC3 capability | mem_uc3_decompose | 1.439 | 1.439 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc3_decompose_1/artifacts.jsonl#reasoning:capability:0:30672` |
| use_cases_live_deep_budgeted_20260614_012157 | UC3 capability | mem_uc3_terse | 0.968 | 0.968 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc3_terse_0/artifacts.jsonl#reasoning:capability:0:99191` |
| use_cases_live_deep_budgeted_20260614_012157 | UC3 capability | mem_uc3_terse | 0.843 | 0.843 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc3_terse_1/artifacts.jsonl#reasoning:capability:0:42611` |
| use_cases_live_deep_budgeted_20260614_012157 | UC3 capability | mem_uc3_verify | 1.441 | 1.441 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc3_verify_0/artifacts.jsonl#reasoning:capability:0:31701` |
| use_cases_live_deep_budgeted_20260614_012157 | UC3 capability | mem_uc3_verify | 1.441 | 1.441 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc3_verify_1/artifacts.jsonl#reasoning:capability:0:7679` |
| use_cases_live_deep_budgeted_20260614_012157 | UC4 family/transfer | mem_uc4_o2_policy | 0.417 | 0.420 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc4_o2_policy_0/artifacts.jsonl#<multi>:policy:0:98666` |
| use_cases_live_deep_budgeted_20260614_012157 | UC4 family/transfer | mem_uc4_o2_policy | 0.421 | 0.421 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc4_o2_policy_1/artifacts.jsonl#<multi>:policy:0:19609` |
| use_cases_live_deep_budgeted_20260614_012157 | UC4 family/transfer | mem_uc4_o3_cold | 0.419 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc4_o3_cold_0/artifacts.jsonl#<holdout>:prior:0:24095` |
| use_cases_live_deep_budgeted_20260614_012157 | UC4 family/transfer | mem_uc4_o3_cold | 0.419 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc4_o3_cold_1/artifacts.jsonl#<holdout>:prior:0:79965` |
| use_cases_live_deep_budgeted_20260614_012157 | UC4 family/transfer | mem_uc4_o3_warm | 0.421 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc4_o3_warm_0/artifacts.jsonl#<holdout>:prior:0:52778` |
| use_cases_live_deep_budgeted_20260614_012157 | UC4 family/transfer | mem_uc4_o3_warm | 0.416 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc4_o3_warm_1/artifacts.jsonl#<holdout>:prior:0:12148` |
| use_cases_live_deep_budgeted_20260614_012157 | UC5 optimizer/tool | mem_uc5_agentic_note | -0.163 | -0.163 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_agentic_note_0/artifacts.jsonl#reasoning:config:0:93443` |
| use_cases_live_deep_budgeted_20260614_012157 | UC5 optimizer/tool | mem_uc5_agentic_note | -0.164 | -0.164 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_agentic_note_1/artifacts.jsonl#reasoning:config:0:47717` |
| use_cases_live_deep_budgeted_20260614_012157 | UC5 optimizer/tool | mem_uc5_agentic_trace_note | -0.165 | -0.165 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_agentic_trace_note_0/artifacts.jsonl#reasoning:config:0:10866` |
| use_cases_live_deep_budgeted_20260614_012157 | UC5 optimizer/tool | mem_uc5_agentic_trace_note | -0.160 | -0.160 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_agentic_trace_note_1/artifacts.jsonl#reasoning:config:0:64119` |
| use_cases_live_deep_budgeted_20260614_012157 | UC5 optimizer/tool | mem_uc5_code_stride | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:25130` |
| use_cases_live_deep_budgeted_20260614_012157 | UC5 optimizer/tool | mem_uc5_code_stride | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_code_stride_1/artifacts.jsonl#internal:batch_design:code:0:26968` |
| use_cases_live_deep_budgeted_20260614_012157 | UC5 optimizer/tool | mem_uc5_code_take_first | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_code_take_first_0/artifacts.jsonl#internal:batch_design:code:1:15406` |
| use_cases_live_deep_budgeted_20260614_012157 | UC5 optimizer/tool | mem_uc5_code_take_first | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_code_take_first_1/artifacts.jsonl#internal:batch_design:code:1:18543` |
| use_cases_live_deep_budgeted_20260614_012157 | UC5 optimizer/tool | mem_uc5_code_take_last | 0.700 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_code_take_last_0/artifacts.jsonl#internal:batch_design:code:1:21749` |
| use_cases_live_deep_budgeted_20260614_012157 | UC5 optimizer/tool | mem_uc5_code_take_last | 0.700 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc5_code_take_last_1/artifacts.jsonl#internal:batch_design:code:1:25083` |
| use_cases_live_deep_budgeted_20260614_012157 | UC6 trace feedback | mem_uc6_trace_hybrid | -0.139 | -0.139 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc6_trace_hybrid_0/artifacts.jsonl#reasoning:config:0:9710` |
| use_cases_live_deep_budgeted_20260614_012157 | UC6 trace feedback | mem_uc6_trace_hybrid | -0.138 | -0.138 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc6_trace_hybrid_1/artifacts.jsonl#reasoning:config:0:77200` |
| use_cases_live_deep_budgeted_20260614_012157 | UC6 trace feedback | mem_uc6_trace_internal | -0.137 | -0.137 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc6_trace_internal_0/artifacts.jsonl#reasoning:config:0:41940` |
| use_cases_live_deep_budgeted_20260614_012157 | UC6 trace feedback | mem_uc6_trace_internal | -0.118 | -0.118 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc6_trace_internal_1/artifacts.jsonl#reasoning:config:0:3911` |
| use_cases_live_deep_budgeted_20260614_012157 | UC6 trace feedback | mem_uc6_trace_otel | -0.139 | -0.139 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc6_trace_otel_0/artifacts.jsonl#reasoning:config:0:76288` |
| use_cases_live_deep_budgeted_20260614_012157 | UC6 trace feedback | mem_uc6_trace_otel | -0.145 | -0.145 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_budgeted_20260614_012157/mem_uc6_trace_otel_1/artifacts.jsonl#reasoning:config:0:37919` |
| use_cases_live_deep_fixed_20260614_004207 | UC1 component code | mem_uc1_batch_design | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:70343` |
| use_cases_live_deep_fixed_20260614_004207 | UC1 component code | mem_uc1_batch_design | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc1_batch_design_1/artifacts.jsonl#internal:batch_design:code:1:74111` |
| use_cases_live_deep_fixed_20260614_004207 | UC1 component code | mem_uc1_trace_summarizer_default | 0.750 | 0.918 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc1_trace_summarizer_default_0/artifacts.jsonl#internal:code_param:code:1:77324` |
| use_cases_live_deep_fixed_20260614_004207 | UC1 component code | mem_uc1_trace_summarizer_default | 0.750 | 0.778 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc1_trace_summarizer_default_1/artifacts.jsonl#internal:code_param:code:1:80035` |
| use_cases_live_deep_fixed_20260614_004207 | UC1 component code | mem_uc1_trace_summarizer_strict | 0.750 | 0.823 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc1_trace_summarizer_strict_0/artifacts.jsonl#internal:code_param:code:1:85508` |
| use_cases_live_deep_fixed_20260614_004207 | UC1 component code | mem_uc1_trace_summarizer_strict | 0.750 | 0.823 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc1_trace_summarizer_strict_1/artifacts.jsonl#internal:code_param:code:1:88179` |
| use_cases_live_deep_fixed_20260614_004207 | UC2 setup/config | mem_uc2_artifact_knowledge | -0.162 | -0.162 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc2_artifact_knowledge_0/artifacts.jsonl#reasoning:config:0:9747` |
| use_cases_live_deep_fixed_20260614_004207 | UC2 setup/config | mem_uc2_artifact_knowledge | -0.152 | -0.152 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc2_artifact_knowledge_1/artifacts.jsonl#reasoning:config:0:79172` |
| use_cases_live_deep_fixed_20260614_004207 | UC2 setup/config | mem_uc2_artifact_only | -0.146 | -0.146 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc2_artifact_only_0/artifacts.jsonl#reasoning:config:0:63828` |
| use_cases_live_deep_fixed_20260614_004207 | UC2 setup/config | mem_uc2_artifact_only | -0.143 | -0.143 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc2_artifact_only_1/artifacts.jsonl#reasoning:config:0:34862` |
| use_cases_live_deep_fixed_20260614_004207 | UC2 setup/config | mem_uc2_warm_prior | -0.149 | -0.149 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc2_warm_prior_0/artifacts.jsonl#reasoning:config:0:59970` |
| use_cases_live_deep_fixed_20260614_004207 | UC2 setup/config | mem_uc2_warm_prior | -0.151 | -0.151 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc2_warm_prior_1/artifacts.jsonl#reasoning:config:0:26635` |
| use_cases_live_deep_fixed_20260614_004207 | UC4 family/transfer | mem_uc4_o2_policy | 0.419 | 0.420 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc4_o2_policy_0/artifacts.jsonl#<multi>:policy:0:32467` |
| use_cases_live_deep_fixed_20260614_004207 | UC4 family/transfer | mem_uc4_o2_policy | 0.422 | 0.422 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc4_o2_policy_1/artifacts.jsonl#<multi>:policy:0:42231` |
| use_cases_live_deep_fixed_20260614_004207 | UC4 family/transfer | mem_uc4_o3_cold | 0.419 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc4_o3_cold_0/artifacts.jsonl#<holdout>:prior:0:61920` |
| use_cases_live_deep_fixed_20260614_004207 | UC4 family/transfer | mem_uc4_o3_cold | 0.419 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc4_o3_cold_1/artifacts.jsonl#<holdout>:prior:0:25003` |
| use_cases_live_deep_fixed_20260614_004207 | UC4 family/transfer | mem_uc4_o3_warm | 0.420 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc4_o3_warm_0/artifacts.jsonl#<holdout>:prior:0:85225` |
| use_cases_live_deep_fixed_20260614_004207 | UC4 family/transfer | mem_uc4_o3_warm | 0.417 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc4_o3_warm_1/artifacts.jsonl#<holdout>:prior:0:56799` |
| use_cases_live_deep_fixed_20260614_004207 | UC5 optimizer/tool | mem_uc5_agentic_note | -0.161 | -0.161 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc5_agentic_note_0/artifacts.jsonl#reasoning:config:0:53170` |
| use_cases_live_deep_fixed_20260614_004207 | UC5 optimizer/tool | mem_uc5_agentic_note | -0.162 | -0.162 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc5_agentic_note_1/artifacts.jsonl#reasoning:config:0:10926` |
| use_cases_live_deep_fixed_20260614_004207 | UC5 optimizer/tool | mem_uc5_agentic_trace_note | -0.166 | -0.166 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc5_agentic_trace_note_0/artifacts.jsonl#reasoning:config:0:80451` |
| use_cases_live_deep_fixed_20260614_004207 | UC5 optimizer/tool | mem_uc5_agentic_trace_note | -0.163 | -0.163 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc5_agentic_trace_note_1/artifacts.jsonl#reasoning:config:0:37494` |
| use_cases_live_deep_fixed_20260614_004207 | UC5 optimizer/tool | mem_uc5_code_stride | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:74840` |
| use_cases_live_deep_fixed_20260614_004207 | UC5 optimizer/tool | mem_uc5_code_stride | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc5_code_stride_1/artifacts.jsonl#internal:batch_design:code:0:76911` |
| use_cases_live_deep_fixed_20260614_004207 | UC5 optimizer/tool | mem_uc5_code_take_first | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc5_code_take_first_0/artifacts.jsonl#internal:batch_design:code:1:63633` |
| use_cases_live_deep_fixed_20260614_004207 | UC5 optimizer/tool | mem_uc5_code_take_first | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc5_code_take_first_1/artifacts.jsonl#internal:batch_design:code:1:67524` |
| use_cases_live_deep_fixed_20260614_004207 | UC5 optimizer/tool | mem_uc5_code_take_last | 0.700 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc5_code_take_last_0/artifacts.jsonl#internal:batch_design:code:1:70729` |
| use_cases_live_deep_fixed_20260614_004207 | UC5 optimizer/tool | mem_uc5_code_take_last | 0.700 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc5_code_take_last_1/artifacts.jsonl#internal:batch_design:code:1:74590` |
| use_cases_live_deep_fixed_20260614_004207 | UC6 trace feedback | mem_uc6_trace_hybrid | -0.116 | -0.116 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc6_trace_hybrid_0/artifacts.jsonl#reasoning:config:0:46689` |
| use_cases_live_deep_fixed_20260614_004207 | UC6 trace feedback | mem_uc6_trace_hybrid | -0.142 | -0.142 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc6_trace_hybrid_1/artifacts.jsonl#reasoning:config:0:18677` |
| use_cases_live_deep_fixed_20260614_004207 | UC6 trace feedback | mem_uc6_trace_internal | -0.143 | -0.143 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc6_trace_internal_0/artifacts.jsonl#reasoning:config:0:44901` |
| use_cases_live_deep_fixed_20260614_004207 | UC6 trace feedback | mem_uc6_trace_internal | -0.144 | -0.144 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc6_trace_internal_1/artifacts.jsonl#reasoning:config:0:19053` |
| use_cases_live_deep_fixed_20260614_004207 | UC6 trace feedback | mem_uc6_trace_otel | -0.152 | -0.152 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc6_trace_otel_0/artifacts.jsonl#reasoning:config:0:98606` |
| use_cases_live_deep_fixed_20260614_004207 | UC6 trace feedback | mem_uc6_trace_otel | -0.139 | -0.139 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_deep_fixed_20260614_004207/mem_uc6_trace_otel_1/artifacts.jsonl#reasoning:config:0:68107` |
| use_cases_live_fixed_20260613_221437 | UC1 component code | mem_uc1_batch_design | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:0:85257` |
| use_cases_live_fixed_20260613_221437 | UC1 component code | mem_uc1_batch_design | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc1_batch_design_1/artifacts.jsonl#internal:batch_design:code:0:89062` |
| use_cases_live_fixed_20260613_221437 | UC1 component code | mem_uc1_batch_design | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc1_batch_design_2/artifacts.jsonl#internal:batch_design:code:0:92647` |
| use_cases_live_fixed_20260613_221437 | UC1 component code | mem_uc1_trace_summarizer | 0.750 | 0.923 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc1_trace_summarizer_0/artifacts.jsonl#internal:code_param:code:15:4019` |
| use_cases_live_fixed_20260613_221437 | UC1 component code | mem_uc1_trace_summarizer | 0.750 | 0.918 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc1_trace_summarizer_1/artifacts.jsonl#internal:code_param:code:0:98778` |
| use_cases_live_fixed_20260613_221437 | UC1 component code | mem_uc1_trace_summarizer | 0.750 | 0.917 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc1_trace_summarizer_2/artifacts.jsonl#internal:code_param:code:15:12592` |
| use_cases_live_fixed_20260613_221437 | UC2 setup/config | mem_uc2 | -0.136 | -0.136 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc2_0/artifacts.jsonl#reasoning:config:0:56210` |
| use_cases_live_fixed_20260613_221437 | UC2 setup/config | mem_uc2 | -0.145 | -0.116 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc2_1/artifacts.jsonl#reasoning:config:2:64082` |
| use_cases_live_fixed_20260613_221437 | UC2 setup/config | mem_uc2 | -0.148 | -0.143 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc2_2/artifacts.jsonl#reasoning:config:2:8983` |
| use_cases_live_fixed_20260613_221437 | UC3 capability | mem_uc3 | 0.968 | 0.968 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc3_0/artifacts.jsonl#reasoning:capability:0:31303` |
| use_cases_live_fixed_20260613_221437 | UC3 capability | mem_uc3 | 0.968 | 0.968 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc3_1/artifacts.jsonl#reasoning:capability:0:49430` |
| use_cases_live_fixed_20260613_221437 | UC3 capability | mem_uc3 | 0.962 | 0.962 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc3_2/artifacts.jsonl#reasoning:capability:0:3643` |
| use_cases_live_fixed_20260613_221437 | UC4 family/transfer | mem_uc4_o2 | 0.418 | 0.422 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc4_o2_0/artifacts.jsonl#<multi>:policy:0:19720` |
| use_cases_live_fixed_20260613_221437 | UC4 family/transfer | mem_uc4_o2 | 0.420 | 0.422 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc4_o2_1/artifacts.jsonl#<multi>:policy:0:65092` |
| use_cases_live_fixed_20260613_221437 | UC4 family/transfer | mem_uc4_o2 | 0.421 | 0.422 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc4_o2_2/artifacts.jsonl#<multi>:policy:0:20384` |
| use_cases_live_fixed_20260613_221437 | UC4 family/transfer | mem_uc4_o3 | 0.421 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc4_o3_0/artifacts.jsonl#<holdout>:prior:11:76577` |
| use_cases_live_fixed_20260613_221437 | UC4 family/transfer | mem_uc4_o3 | 0.422 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc4_o3_1/artifacts.jsonl#<holdout>:prior:11:19697` |
| use_cases_live_fixed_20260613_221437 | UC4 family/transfer | mem_uc4_o3 | 0.422 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc4_o3_2/artifacts.jsonl#<holdout>:prior:0:31092` |
| use_cases_live_fixed_20260613_221437 | UC5 optimizer/tool | mem_uc5_stride | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc5_stride_0/artifacts.jsonl#internal:batch_design:code:0:86973` |
| use_cases_live_fixed_20260613_221437 | UC5 optimizer/tool | mem_uc5_stride | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc5_stride_1/artifacts.jsonl#internal:batch_design:code:0:89221` |
| use_cases_live_fixed_20260613_221437 | UC5 optimizer/tool | mem_uc5_stride | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc5_stride_2/artifacts.jsonl#internal:batch_design:code:0:91271` |
| use_cases_live_fixed_20260613_221437 | UC5 optimizer/tool | mem_uc5_take_first | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc5_take_first_0/artifacts.jsonl#internal:batch_design:code:0:58574` |
| use_cases_live_fixed_20260613_221437 | UC5 optimizer/tool | mem_uc5_take_first | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc5_take_first_1/artifacts.jsonl#internal:batch_design:code:0:62056` |
| use_cases_live_fixed_20260613_221437 | UC5 optimizer/tool | mem_uc5_take_first | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc5_take_first_2/artifacts.jsonl#internal:batch_design:code:0:64932` |
| use_cases_live_fixed_20260613_221437 | UC5 optimizer/tool | mem_uc5_take_last | 0.700 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc5_take_last_0/artifacts.jsonl#internal:batch_design:code:0:68401` |
| use_cases_live_fixed_20260613_221437 | UC5 optimizer/tool | mem_uc5_take_last | 0.700 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc5_take_last_1/artifacts.jsonl#internal:batch_design:code:0:83740` |
| use_cases_live_fixed_20260613_221437 | UC5 optimizer/tool | mem_uc5_take_last | 0.700 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc5_take_last_2/artifacts.jsonl#internal:batch_design:code:0:86940` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_ch_episode | -0.151 | -0.151 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_ch_episode_0/artifacts.jsonl#reasoning:config:0:54569` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_ch_episode | -0.145 | -0.145 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_ch_episode_1/artifacts.jsonl#reasoning:config:0:92183` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_ch_episode | -0.135 | -0.135 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_ch_episode_2/artifacts.jsonl#reasoning:config:0:31048` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_ch_full | -0.140 | -0.140 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_ch_full_0/artifacts.jsonl#reasoning:config:0:71024` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_ch_full | -0.144 | -0.144 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_ch_full_1/artifacts.jsonl#reasoning:config:0:11108` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_ch_full | -0.138 | -0.138 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_ch_full_2/artifacts.jsonl#reasoning:config:0:50122` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_ch_step | -0.120 | -0.120 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_ch_step_0/artifacts.jsonl#reasoning:config:0:28025` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_ch_step | -0.143 | -0.143 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_ch_step_1/artifacts.jsonl#reasoning:config:0:68733` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_ch_step | -0.144 | -0.144 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_ch_step_2/artifacts.jsonl#reasoning:config:0:14498` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_joint | -0.145 | -0.145 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_joint_0/artifacts.jsonl#reasoning:config:0:92425` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_joint | -0.150 | -0.150 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_joint_1/artifacts.jsonl#reasoning:config:0:33986` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_joint | -0.152 | -0.152 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_joint_2/artifacts.jsonl#reasoning:config:0:92047` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_tt_hybrid | -0.146 | -0.146 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_tt_hybrid_0/artifacts.jsonl#reasoning:config:0:94652` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_tt_hybrid | -0.146 | -0.146 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_tt_hybrid_1/artifacts.jsonl#reasoning:config:0:37860` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_tt_hybrid | -0.150 | -0.150 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_tt_hybrid_2/artifacts.jsonl#reasoning:config:0:87012` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_tt_internal | -0.151 | -0.151 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_tt_internal_0/artifacts.jsonl#reasoning:config:0:43250` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_tt_internal | -0.124 | -0.124 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_tt_internal_1/artifacts.jsonl#reasoning:config:0:82571` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_tt_internal | -0.153 | -0.153 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_tt_internal_2/artifacts.jsonl#reasoning:config:0:22723` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_tt_otel | -0.137 | -0.137 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_tt_otel_0/artifacts.jsonl#reasoning:config:0:61833` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_tt_otel | -0.154 | -0.154 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_tt_otel_1/artifacts.jsonl#reasoning:config:0:11701` |
| use_cases_live_fixed_20260613_221437 | UC6 trace feedback | mem_uc6_o1_tt_otel | -0.155 | -0.155 | `notebook_outputs/recursive_opt_use_cases/use_cases_live_fixed_20260613_221437/mem_uc6_o1_tt_otel_2/artifacts.jsonl#reasoning:config:0:53582` |
| use_cases_rootcause_20260614_022600 | UC1 component code | mem_uc1_batch_design | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:99106` |
| use_cases_rootcause_20260614_022600 | UC1 component code | mem_uc1_batch_design | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc1_batch_design_1/artifacts.jsonl#internal:batch_design:code:1:2381` |
| use_cases_rootcause_20260614_022600 | UC1 component code | mem_uc1_bbeh_direct_solver | 0.625 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc1_bbeh_direct_solver_0/artifacts.jsonl#internal:multiobjective_bbeh:code:1:19492` |
| use_cases_rootcause_20260614_022600 | UC1 component code | mem_uc1_bbeh_direct_solver | 0.625 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc1_bbeh_direct_solver_1/artifacts.jsonl#internal:multiobjective_bbeh:code:1:24517` |
| use_cases_rootcause_20260614_022600 | UC1 component code | mem_uc1_trace_summarizer_default | 0.750 | 0.876 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc1_trace_summarizer_default_0/artifacts.jsonl#internal:code_param:code:1:5347` |
| use_cases_rootcause_20260614_022600 | UC1 component code | mem_uc1_trace_summarizer_default | 0.750 | 0.823 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc1_trace_summarizer_default_1/artifacts.jsonl#internal:code_param:code:1:7919` |
| use_cases_rootcause_20260614_022600 | UC1 component code | mem_uc1_trace_summarizer_strict | 0.750 | 0.876 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc1_trace_summarizer_strict_0/artifacts.jsonl#internal:code_param:code:1:11703` |
| use_cases_rootcause_20260614_022600 | UC1 component code | mem_uc1_trace_summarizer_strict | 0.750 | 0.823 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc1_trace_summarizer_strict_1/artifacts.jsonl#internal:code_param:code:1:14499` |
| use_cases_rootcause_20260614_022600 | UC2 setup/config | mem_uc2_artifact_knowledge | -0.158 | -0.158 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc2_artifact_knowledge_0/artifacts.jsonl#reasoning:config:0:88784` |
| use_cases_rootcause_20260614_022600 | UC2 setup/config | mem_uc2_artifact_knowledge | -0.161 | -0.161 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc2_artifact_knowledge_1/artifacts.jsonl#reasoning:config:0:73571` |
| use_cases_rootcause_20260614_022600 | UC2 setup/config | mem_uc2_artifact_only | -0.117 | -0.117 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc2_artifact_only_0/artifacts.jsonl#reasoning:config:0:18595` |
| use_cases_rootcause_20260614_022600 | UC2 setup/config | mem_uc2_artifact_only | -0.150 | -0.150 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc2_artifact_only_1/artifacts.jsonl#reasoning:config:0:99493` |
| use_cases_rootcause_20260614_022600 | UC2 setup/config | mem_uc2_drop | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc2_drop_0/artifacts.jsonl#drop:config:0:81549` |
| use_cases_rootcause_20260614_022600 | UC2 setup/config | mem_uc2_drop | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc2_drop_1/artifacts.jsonl#drop:config:0:22408` |
| use_cases_rootcause_20260614_022600 | UC2 setup/config | mem_uc2_qasper | 0.165 | 0.165 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc2_qasper_0/artifacts.jsonl#qasper:config:0:74219` |
| use_cases_rootcause_20260614_022600 | UC2 setup/config | mem_uc2_qasper | 0.159 | 0.159 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc2_qasper_1/artifacts.jsonl#qasper:config:0:10879` |
| use_cases_rootcause_20260614_022600 | UC2 setup/config | mem_uc2_warm_prior | -0.131 | -0.131 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc2_warm_prior_0/artifacts.jsonl#reasoning:config:0:60204` |
| use_cases_rootcause_20260614_022600 | UC2 setup/config | mem_uc2_warm_prior | -0.123 | -0.123 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc2_warm_prior_1/artifacts.jsonl#reasoning:config:0:37002` |
| use_cases_rootcause_20260614_022600 | UC3 capability | mem_uc3_decompose | 1.439 | 1.439 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc3_decompose_0/artifacts.jsonl#reasoning:capability:0:97133` |
| use_cases_rootcause_20260614_022600 | UC3 capability | mem_uc3_decompose | 1.439 | 1.439 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc3_decompose_1/artifacts.jsonl#reasoning:capability:0:59495` |
| use_cases_rootcause_20260614_022600 | UC3 capability | mem_uc3_terse | 0.968 | 0.968 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc3_terse_0/artifacts.jsonl#reasoning:capability:0:80614` |
| use_cases_rootcause_20260614_022600 | UC3 capability | mem_uc3_terse | 1.184 | 1.184 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc3_terse_1/artifacts.jsonl#reasoning:capability:0:43897` |
| use_cases_rootcause_20260614_022600 | UC3 capability | mem_uc3_verify | 1.441 | 1.441 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc3_verify_0/artifacts.jsonl#reasoning:capability:0:32066` |
| use_cases_rootcause_20260614_022600 | UC3 capability | mem_uc3_verify | 1.441 | 1.441 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc3_verify_1/artifacts.jsonl#reasoning:capability:0:8865` |
| use_cases_rootcause_20260614_022600 | UC4 family/transfer | mem_uc4_o2_policy | 0.084 | 0.084 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc4_o2_policy_0/artifacts.jsonl#<multi>:policy:0:27286` |
| use_cases_rootcause_20260614_022600 | UC4 family/transfer | mem_uc4_o2_policy | 0.124 | 0.124 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc4_o2_policy_1/artifacts.jsonl#<multi>:policy:0:29686` |
| use_cases_rootcause_20260614_022600 | UC4 family/transfer | mem_uc4_o3_cold | 0.129 | 0.129 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc4_o3_cold_0/artifacts.jsonl#<multi>:policy:0:56150` |
| use_cases_rootcause_20260614_022600 | UC4 family/transfer | mem_uc4_o3_cold | 0.005 | 0.036 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc4_o3_cold_1/artifacts.jsonl#<holdout>:prior:0:43077` |
| use_cases_rootcause_20260614_022600 | UC4 family/transfer | mem_uc4_o3_warm | -0.002 | 0.037 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc4_o3_warm_0/artifacts.jsonl#<holdout>:prior:0:18024` |
| use_cases_rootcause_20260614_022600 | UC4 family/transfer | mem_uc4_o3_warm | -0.011 | 0.043 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc4_o3_warm_1/artifacts.jsonl#<holdout>:prior:0:69475` |
| use_cases_rootcause_20260614_022600 | UC5 optimizer/tool | mem_uc5_agentic_note | -0.164 | -0.164 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc5_agentic_note_0/artifacts.jsonl#reasoning:config:0:7408` |
| use_cases_rootcause_20260614_022600 | UC5 optimizer/tool | mem_uc5_agentic_note | -0.171 | -0.171 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc5_agentic_note_1/artifacts.jsonl#reasoning:config:0:74992` |
| use_cases_rootcause_20260614_022600 | UC5 optimizer/tool | mem_uc5_agentic_trace_note | -0.163 | -0.163 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc5_agentic_trace_note_0/artifacts.jsonl#reasoning:config:0:54866` |
| use_cases_rootcause_20260614_022600 | UC5 optimizer/tool | mem_uc5_agentic_trace_note | -0.164 | -0.164 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc5_agentic_trace_note_1/artifacts.jsonl#reasoning:config:0:22754` |
| use_cases_rootcause_20260614_022600 | UC5 optimizer/tool | mem_uc5_code_stride | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:22248` |
| use_cases_rootcause_20260614_022600 | UC5 optimizer/tool | mem_uc5_code_stride | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc5_code_stride_1/artifacts.jsonl#internal:batch_design:code:0:24822` |
| use_cases_rootcause_20260614_022600 | UC5 optimizer/tool | mem_uc5_code_take_first | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc5_code_take_first_0/artifacts.jsonl#internal:batch_design:code:1:9733` |
| use_cases_rootcause_20260614_022600 | UC5 optimizer/tool | mem_uc5_code_take_first | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc5_code_take_first_1/artifacts.jsonl#internal:batch_design:code:1:14035` |
| use_cases_rootcause_20260614_022600 | UC5 optimizer/tool | mem_uc5_code_take_last | 0.700 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc5_code_take_last_0/artifacts.jsonl#internal:batch_design:code:1:18298` |
| use_cases_rootcause_20260614_022600 | UC5 optimizer/tool | mem_uc5_code_take_last | 0.700 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc5_code_take_last_1/artifacts.jsonl#internal:batch_design:code:1:22208` |
| use_cases_rootcause_20260614_022600 | UC6 trace feedback | mem_uc6_qasper_trace_hybrid | 0.099 | 0.099 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc6_qasper_trace_hybrid_0/artifacts.jsonl#reasoning:config:0:55592` |
| use_cases_rootcause_20260614_022600 | UC6 trace feedback | mem_uc6_qasper_trace_hybrid | 0.155 | 0.155 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc6_qasper_trace_hybrid_1/artifacts.jsonl#reasoning:config:0:1672` |
| use_cases_rootcause_20260614_022600 | UC6 trace feedback | mem_uc6_qasper_trace_internal | 0.120 | 0.120 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc6_qasper_trace_internal_0/artifacts.jsonl#reasoning:config:0:77415` |
| use_cases_rootcause_20260614_022600 | UC6 trace feedback | mem_uc6_qasper_trace_internal | 0.123 | 0.123 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc6_qasper_trace_internal_1/artifacts.jsonl#reasoning:config:0:12130` |
| use_cases_rootcause_20260614_022600 | UC6 trace feedback | mem_uc6_qasper_trace_otel | 0.164 | 0.164 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc6_qasper_trace_otel_0/artifacts.jsonl#reasoning:config:0:60364` |
| use_cases_rootcause_20260614_022600 | UC6 trace feedback | mem_uc6_qasper_trace_otel | 0.142 | 0.142 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc6_qasper_trace_otel_1/artifacts.jsonl#reasoning:config:0:7978` |
| use_cases_rootcause_20260614_022600 | UC6 trace feedback | mem_uc6_trace_hybrid | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc6_trace_hybrid_0/artifacts.jsonl#reasoning:config:0:24217` |
| use_cases_rootcause_20260614_022600 | UC6 trace feedback | mem_uc6_trace_hybrid | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc6_trace_hybrid_1/artifacts.jsonl#reasoning:config:0:66367` |
| use_cases_rootcause_20260614_022600 | UC6 trace feedback | mem_uc6_trace_internal | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc6_trace_internal_0/artifacts.jsonl#reasoning:config:0:70987` |
| use_cases_rootcause_20260614_022600 | UC6 trace feedback | mem_uc6_trace_internal | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc6_trace_internal_1/artifacts.jsonl#reasoning:config:0:16453` |
| use_cases_rootcause_20260614_022600 | UC6 trace feedback | mem_uc6_trace_otel | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc6_trace_otel_0/artifacts.jsonl#reasoning:config:0:53725` |
| use_cases_rootcause_20260614_022600 | UC6 trace feedback | mem_uc6_trace_otel | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_20260614_022600/mem_uc6_trace_otel_1/artifacts.jsonl#reasoning:config:0:85878` |
| use_cases_rootcause_final_20260614 | UC1 component code | mem_uc1_batch_design | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:30451` |
| use_cases_rootcause_final_20260614 | UC1 component code | mem_uc1_batch_design | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc1_batch_design_1/artifacts.jsonl#internal:batch_design:code:1:34245` |
| use_cases_rootcause_final_20260614 | UC1 component code | mem_uc1_bbeh_direct_solver | 0.625 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc1_bbeh_direct_solver_0/artifacts.jsonl#internal:multiobjective_bbeh:code:1:54006` |
| use_cases_rootcause_final_20260614 | UC1 component code | mem_uc1_bbeh_direct_solver | 0.625 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc1_bbeh_direct_solver_1/artifacts.jsonl#internal:multiobjective_bbeh:code:1:58776` |
| use_cases_rootcause_final_20260614 | UC1 component code | mem_uc1_trace_summarizer_default | 0.750 | 0.876 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc1_trace_summarizer_default_0/artifacts.jsonl#internal:code_param:code:1:36904` |
| use_cases_rootcause_final_20260614 | UC1 component code | mem_uc1_trace_summarizer_default | 0.750 | 0.918 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc1_trace_summarizer_default_1/artifacts.jsonl#internal:code_param:code:1:40708` |
| use_cases_rootcause_final_20260614 | UC1 component code | mem_uc1_trace_summarizer_strict | 0.750 | 0.918 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc1_trace_summarizer_strict_0/artifacts.jsonl#internal:code_param:code:1:44914` |
| use_cases_rootcause_final_20260614 | UC1 component code | mem_uc1_trace_summarizer_strict | 0.750 | 0.876 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc1_trace_summarizer_strict_1/artifacts.jsonl#internal:code_param:code:1:48892` |
| use_cases_rootcause_final_20260614 | UC2 setup/config | mem_uc2_artifact_knowledge | -0.165 | -0.165 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_artifact_knowledge_0/artifacts.jsonl#reasoning:config:0:25460` |
| use_cases_rootcause_final_20260614 | UC2 setup/config | mem_uc2_artifact_knowledge | -0.157 | -0.157 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_artifact_knowledge_1/artifacts.jsonl#reasoning:config:0:719` |
| use_cases_rootcause_final_20260614 | UC2 setup/config | mem_uc2_artifact_only | -0.147 | -0.147 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_artifact_only_0/artifacts.jsonl#reasoning:config:0:52501` |
| use_cases_rootcause_final_20260614 | UC2 setup/config | mem_uc2_artifact_only | -0.125 | -0.125 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_artifact_only_1/artifacts.jsonl#reasoning:config:0:30223` |
| use_cases_rootcause_final_20260614 | UC2 setup/config | mem_uc2_drop | 0.750 | 0.750 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_drop_0/artifacts.jsonl#drop:config:0:22041` |
| use_cases_rootcause_final_20260614 | UC2 setup/config | mem_uc2_drop | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_drop_1/artifacts.jsonl#drop:config:0:60919` |
| use_cases_rootcause_final_20260614 | UC2 setup/config | mem_uc2_qasper | 0.143 | 0.143 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_qasper_0/artifacts.jsonl#qasper:config:0:20308` |
| use_cases_rootcause_final_20260614 | UC2 setup/config | mem_uc2_qasper | 0.183 | 0.183 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_qasper_1/artifacts.jsonl#qasper:config:0:62209` |
| use_cases_rootcause_final_20260614 | UC2 setup/config | mem_uc2_warm_prior | -0.116 | -0.116 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_warm_prior_0/artifacts.jsonl#reasoning:config:0:87661` |
| use_cases_rootcause_final_20260614 | UC2 setup/config | mem_uc2_warm_prior | -0.152 | -0.152 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc2_warm_prior_1/artifacts.jsonl#reasoning:config:0:66610` |
| use_cases_rootcause_final_20260614 | UC3 capability | mem_uc3_decompose | 1.451 | 1.451 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc3_decompose_0/artifacts.jsonl#reasoning:capability:0:59196` |
| use_cases_rootcause_final_20260614 | UC3 capability | mem_uc3_decompose | 1.439 | 1.439 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc3_decompose_1/artifacts.jsonl#reasoning:capability:0:35486` |
| use_cases_rootcause_final_20260614 | UC3 capability | mem_uc3_terse | 0.968 | 0.968 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc3_terse_0/artifacts.jsonl#reasoning:capability:0:38889` |
| use_cases_rootcause_final_20260614 | UC3 capability | mem_uc3_terse | 0.788 | 0.788 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc3_terse_1/artifacts.jsonl#reasoning:capability:0:98998` |
| use_cases_rootcause_final_20260614 | UC3 capability | mem_uc3_verify | 1.441 | 1.441 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc3_verify_0/artifacts.jsonl#reasoning:capability:0:96998` |
| use_cases_rootcause_final_20260614 | UC3 capability | mem_uc3_verify | 1.316 | 1.316 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc3_verify_1/artifacts.jsonl#reasoning:capability:0:69414` |
| use_cases_rootcause_final_20260614 | UC4 family/transfer | mem_uc4_o2_policy | -0.020 | 0.028 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc4_o2_policy_0/artifacts.jsonl#<multi>:policy:0:89297` |
| use_cases_rootcause_final_20260614 | UC4 family/transfer | mem_uc4_o2_policy | 0.085 | 0.101 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc4_o2_policy_1/artifacts.jsonl#<multi>:policy:0:44445` |
| use_cases_rootcause_final_20260614 | UC4 family/transfer | mem_uc4_o3_cold | 0.021 | 0.021 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc4_o3_cold_0/artifacts.jsonl#<multi>:policy:0:14469` |
| use_cases_rootcause_final_20260614 | UC4 family/transfer | mem_uc4_o3_cold | 0.063 | 0.063 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc4_o3_cold_1/artifacts.jsonl#<multi>:policy:0:47652` |
| use_cases_rootcause_final_20260614 | UC4 family/transfer | mem_uc4_o3_warm | 0.021 | 0.075 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc4_o3_warm_0/artifacts.jsonl#<multi>:policy:0:9064` |
| use_cases_rootcause_final_20260614 | UC4 family/transfer | mem_uc4_o3_warm | 0.085 | 0.085 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc4_o3_warm_1/artifacts.jsonl#<multi>:policy:0:25332` |
| use_cases_rootcause_final_20260614 | UC5 optimizer/tool | mem_uc5_agentic_note | -0.158 | -0.158 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_agentic_note_0/artifacts.jsonl#reasoning:config:0:15333` |
| use_cases_rootcause_final_20260614 | UC5 optimizer/tool | mem_uc5_agentic_note | -0.158 | -0.158 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_agentic_note_1/artifacts.jsonl#reasoning:config:0:80152` |
| use_cases_rootcause_final_20260614 | UC5 optimizer/tool | mem_uc5_agentic_trace_note | -0.160 | -0.160 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_agentic_trace_note_0/artifacts.jsonl#reasoning:config:0:6559` |
| use_cases_rootcause_final_20260614 | UC5 optimizer/tool | mem_uc5_agentic_trace_note | -0.158 | -0.158 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_agentic_trace_note_1/artifacts.jsonl#reasoning:config:0:74145` |
| use_cases_rootcause_final_20260614 | UC5 optimizer/tool | mem_uc5_agentic_trace_search | -0.170 | -0.170 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_agentic_trace_search_0/artifacts.jsonl#reasoning:config:0:60152` |
| use_cases_rootcause_final_20260614 | UC5 optimizer/tool | mem_uc5_agentic_trace_search | -0.159 | -0.159 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_agentic_trace_search_1/artifacts.jsonl#reasoning:config:0:24025` |
| use_cases_rootcause_final_20260614 | UC5 optimizer/tool | mem_uc5_code_stride | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:31029` |
| use_cases_rootcause_final_20260614 | UC5 optimizer/tool | mem_uc5_code_stride | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_code_stride_1/artifacts.jsonl#internal:batch_design:code:0:32137` |
| use_cases_rootcause_final_20260614 | UC5 optimizer/tool | mem_uc5_code_take_first | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_code_take_first_0/artifacts.jsonl#internal:batch_design:code:1:20821` |
| use_cases_rootcause_final_20260614 | UC5 optimizer/tool | mem_uc5_code_take_first | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_code_take_first_1/artifacts.jsonl#internal:batch_design:code:1:23818` |
| use_cases_rootcause_final_20260614 | UC5 optimizer/tool | mem_uc5_code_take_last | 0.700 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_code_take_last_0/artifacts.jsonl#internal:batch_design:code:1:27076` |
| use_cases_rootcause_final_20260614 | UC5 optimizer/tool | mem_uc5_code_take_last | 0.700 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc5_code_take_last_1/artifacts.jsonl#internal:batch_design:code:1:30971` |
| use_cases_rootcause_final_20260614 | UC6 trace feedback | mem_uc6_trace_hybrid | 0.188 | 0.188 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc6_trace_hybrid_0/artifacts.jsonl#reasoning:config:0:24828` |
| use_cases_rootcause_final_20260614 | UC6 trace feedback | mem_uc6_trace_hybrid | 0.134 | 0.134 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc6_trace_hybrid_1/artifacts.jsonl#reasoning:config:0:72130` |
| use_cases_rootcause_final_20260614 | UC6 trace feedback | mem_uc6_trace_internal | 0.168 | 0.168 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc6_trace_internal_0/artifacts.jsonl#reasoning:config:0:23037` |
| use_cases_rootcause_final_20260614 | UC6 trace feedback | mem_uc6_trace_internal | 0.195 | 0.195 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc6_trace_internal_1/artifacts.jsonl#reasoning:config:0:69377` |
| use_cases_rootcause_final_20260614 | UC6 trace feedback | mem_uc6_trace_otel | 0.253 | 0.253 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc6_trace_otel_0/artifacts.jsonl#reasoning:config:0:26419` |
| use_cases_rootcause_final_20260614 | UC6 trace feedback | mem_uc6_trace_otel | 0.122 | 0.122 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_final_20260614/mem_uc6_trace_otel_1/artifacts.jsonl#reasoning:config:0:76585` |
| use_cases_rootcause_qasper_20260614_032217 | UC1 component code | mem_uc1_batch_design | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc1_batch_design_0/artifacts.jsonl#internal:batch_design:code:1:65938` |
| use_cases_rootcause_qasper_20260614_032217 | UC1 component code | mem_uc1_batch_design | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc1_batch_design_1/artifacts.jsonl#internal:batch_design:code:1:69824` |
| use_cases_rootcause_qasper_20260614_032217 | UC1 component code | mem_uc1_bbeh_direct_solver | 0.625 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc1_bbeh_direct_solver_0/artifacts.jsonl#internal:multiobjective_bbeh:code:1:86234` |
| use_cases_rootcause_qasper_20260614_032217 | UC1 component code | mem_uc1_bbeh_direct_solver | 0.625 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc1_bbeh_direct_solver_1/artifacts.jsonl#internal:multiobjective_bbeh:code:1:90829` |
| use_cases_rootcause_qasper_20260614_032217 | UC1 component code | mem_uc1_trace_summarizer_default | 0.750 | 0.750 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc1_trace_summarizer_default_0/artifacts.jsonl#internal:code_param:code:0:69862` |
| use_cases_rootcause_qasper_20260614_032217 | UC1 component code | mem_uc1_trace_summarizer_default | 0.750 | 0.823 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc1_trace_summarizer_default_1/artifacts.jsonl#internal:code_param:code:1:75557` |
| use_cases_rootcause_qasper_20260614_032217 | UC1 component code | mem_uc1_trace_summarizer_strict | 0.750 | 0.823 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc1_trace_summarizer_strict_0/artifacts.jsonl#internal:code_param:code:1:79189` |
| use_cases_rootcause_qasper_20260614_032217 | UC1 component code | mem_uc1_trace_summarizer_strict | 0.750 | 0.750 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc1_trace_summarizer_strict_1/artifacts.jsonl#internal:code_param:code:0:79224` |
| use_cases_rootcause_qasper_20260614_032217 | UC2 setup/config | mem_uc2_artifact_knowledge | -0.169 | -0.169 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc2_artifact_knowledge_0/artifacts.jsonl#reasoning:config:0:49265` |
| use_cases_rootcause_qasper_20260614_032217 | UC2 setup/config | mem_uc2_artifact_knowledge | -0.156 | -0.156 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc2_artifact_knowledge_1/artifacts.jsonl#reasoning:config:0:30980` |
| use_cases_rootcause_qasper_20260614_032217 | UC2 setup/config | mem_uc2_artifact_only | -0.141 | -0.141 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc2_artifact_only_0/artifacts.jsonl#reasoning:config:0:86138` |
| use_cases_rootcause_qasper_20260614_032217 | UC2 setup/config | mem_uc2_artifact_only | -0.123 | -0.123 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc2_artifact_only_1/artifacts.jsonl#reasoning:config:0:68470` |
| use_cases_rootcause_qasper_20260614_032217 | UC2 setup/config | mem_uc2_drop | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc2_drop_0/artifacts.jsonl#drop:config:0:49092` |
| use_cases_rootcause_qasper_20260614_032217 | UC2 setup/config | mem_uc2_drop | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc2_drop_1/artifacts.jsonl#drop:config:0:80119` |
| use_cases_rootcause_qasper_20260614_032217 | UC2 setup/config | mem_uc2_qasper | 0.149 | 0.149 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc2_qasper_0/artifacts.jsonl#qasper:config:0:18522` |
| use_cases_rootcause_qasper_20260614_032217 | UC2 setup/config | mem_uc2_qasper | 0.128 | 0.128 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc2_qasper_1/artifacts.jsonl#qasper:config:0:58151` |
| use_cases_rootcause_qasper_20260614_032217 | UC2 setup/config | mem_uc2_warm_prior | -0.124 | -0.124 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc2_warm_prior_0/artifacts.jsonl#reasoning:config:0:22628` |
| use_cases_rootcause_qasper_20260614_032217 | UC2 setup/config | mem_uc2_warm_prior | -0.155 | -0.155 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc2_warm_prior_1/artifacts.jsonl#reasoning:config:0:7210` |
| use_cases_rootcause_qasper_20260614_032217 | UC3 capability | mem_uc3_decompose | 1.448 | 1.448 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc3_decompose_0/artifacts.jsonl#reasoning:capability:0:54745` |
| use_cases_rootcause_qasper_20260614_032217 | UC3 capability | mem_uc3_decompose | 1.439 | 1.439 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc3_decompose_1/artifacts.jsonl#reasoning:capability:0:27551` |
| use_cases_rootcause_qasper_20260614_032217 | UC3 capability | mem_uc3_terse | 0.843 | 0.843 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc3_terse_0/artifacts.jsonl#reasoning:capability:0:24812` |
| use_cases_rootcause_qasper_20260614_032217 | UC3 capability | mem_uc3_terse | 0.968 | 0.968 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc3_terse_1/artifacts.jsonl#reasoning:capability:0:84100` |
| use_cases_rootcause_qasper_20260614_032217 | UC3 capability | mem_uc3_verify | 1.441 | 1.441 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc3_verify_0/artifacts.jsonl#reasoning:capability:0:79334` |
| use_cases_rootcause_qasper_20260614_032217 | UC3 capability | mem_uc3_verify | 1.441 | 1.441 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc3_verify_1/artifacts.jsonl#reasoning:capability:0:65045` |
| use_cases_rootcause_qasper_20260614_032217 | UC4 family/transfer | mem_uc4_o2_policy | 0.092 | 0.092 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc4_o2_policy_0/artifacts.jsonl#<multi>:policy:0:94892` |
| use_cases_rootcause_qasper_20260614_032217 | UC4 family/transfer | mem_uc4_o2_policy | -0.119 | 0.023 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc4_o2_policy_1/artifacts.jsonl#<multi>:policy:0:35530` |
| use_cases_rootcause_qasper_20260614_032217 | UC4 family/transfer | mem_uc4_o3_cold | -0.011 | 0.031 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc4_o3_cold_0/artifacts.jsonl#<holdout>:prior:0:16166` |
| use_cases_rootcause_qasper_20260614_032217 | UC4 family/transfer | mem_uc4_o3_cold | -0.142 | 0.158 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc4_o3_cold_1/artifacts.jsonl#<holdout>:prior:0:5603` |
| use_cases_rootcause_qasper_20260614_032217 | UC4 family/transfer | mem_uc4_o3_warm | -0.117 | 0.039 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc4_o3_warm_0/artifacts.jsonl#<holdout>:prior:0:85262` |
| use_cases_rootcause_qasper_20260614_032217 | UC4 family/transfer | mem_uc4_o3_warm | -0.007 | 0.015 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc4_o3_warm_1/artifacts.jsonl#<multi>:policy:0:31652` |
| use_cases_rootcause_qasper_20260614_032217 | UC5 optimizer/tool | mem_uc5_agentic_note | -0.158 | -0.158 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc5_agentic_note_0/artifacts.jsonl#reasoning:config:0:66921` |
| use_cases_rootcause_qasper_20260614_032217 | UC5 optimizer/tool | mem_uc5_agentic_note | -0.167 | -0.167 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc5_agentic_note_1/artifacts.jsonl#reasoning:config:0:36392` |
| use_cases_rootcause_qasper_20260614_032217 | UC5 optimizer/tool | mem_uc5_agentic_trace_note | -0.165 | -0.165 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc5_agentic_trace_note_0/artifacts.jsonl#reasoning:config:0:16541` |
| use_cases_rootcause_qasper_20260614_032217 | UC5 optimizer/tool | mem_uc5_agentic_trace_note | -0.160 | -0.160 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc5_agentic_trace_note_1/artifacts.jsonl#reasoning:config:0:82321` |
| use_cases_rootcause_qasper_20260614_032217 | UC5 optimizer/tool | mem_uc5_code_stride | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc5_code_stride_0/artifacts.jsonl#internal:batch_design:code:0:79161` |
| use_cases_rootcause_qasper_20260614_032217 | UC5 optimizer/tool | mem_uc5_code_stride | 1.000 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc5_code_stride_1/artifacts.jsonl#internal:batch_design:code:0:82225` |
| use_cases_rootcause_qasper_20260614_032217 | UC5 optimizer/tool | mem_uc5_code_take_first | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc5_code_take_first_0/artifacts.jsonl#internal:batch_design:code:1:68989` |
| use_cases_rootcause_qasper_20260614_032217 | UC5 optimizer/tool | mem_uc5_code_take_first | 0.800 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc5_code_take_first_1/artifacts.jsonl#internal:batch_design:code:1:72405` |
| use_cases_rootcause_qasper_20260614_032217 | UC5 optimizer/tool | mem_uc5_code_take_last | 0.700 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc5_code_take_last_0/artifacts.jsonl#internal:batch_design:code:1:75749` |
| use_cases_rootcause_qasper_20260614_032217 | UC5 optimizer/tool | mem_uc5_code_take_last | 0.700 | 1.000 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc5_code_take_last_1/artifacts.jsonl#internal:batch_design:code:1:79115` |
| use_cases_rootcause_qasper_20260614_032217 | UC6 trace feedback | mem_uc6_trace_hybrid | 0.124 | 0.124 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc6_trace_hybrid_0/artifacts.jsonl#reasoning:config:0:91109` |
| use_cases_rootcause_qasper_20260614_032217 | UC6 trace feedback | mem_uc6_trace_hybrid | 0.152 | 0.152 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc6_trace_hybrid_1/artifacts.jsonl#reasoning:config:0:33914` |
| use_cases_rootcause_qasper_20260614_032217 | UC6 trace feedback | mem_uc6_trace_internal | 0.201 | 0.201 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc6_trace_internal_0/artifacts.jsonl#reasoning:config:0:28292` |
| use_cases_rootcause_qasper_20260614_032217 | UC6 trace feedback | mem_uc6_trace_internal | 0.140 | 0.140 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc6_trace_internal_1/artifacts.jsonl#reasoning:config:0:67810` |
| use_cases_rootcause_qasper_20260614_032217 | UC6 trace feedback | mem_uc6_trace_otel | 0.135 | 0.135 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc6_trace_otel_0/artifacts.jsonl#reasoning:config:0:11036` |
| use_cases_rootcause_qasper_20260614_032217 | UC6 trace feedback | mem_uc6_trace_otel | 0.179 | 0.179 | `notebook_outputs/recursive_opt_use_cases/use_cases_rootcause_qasper_20260614_032217/mem_uc6_trace_otel_1/artifacts.jsonl#reasoning:config:0:44823` |


**Interpretation guardrails:** UC1/UC5 code-helper scores are validator evidence and can saturate; UC2/UC6 are real Trace-Bench prompt/config scores over the configured examples; UC1 now includes a real BBEH direct-code arm to avoid relying only on toy validators; UC3 remains a prompt-capability surface on GSM8K; UC4 transfer is only meaningful when warm beats cold by more than run noise. Reusable artifacts are in each listed `artifacts.jsonl` under the `content` field; code arms save Python code, config arms save config text, and optimizer-tool arms save tool names/evidence hooks.
